# D67 Docker K8s Code

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

last modified date : 2026.11.20
제작 : 박광석 (모두의연구소)

# 컨테이너 심화와 쿠버네티스 구현 실습

## 실습 목표와 핵심 질문

**같은 서비스를 다른 컴퓨터에서도 실행하고 새 버전으로 바꾸려면 무엇을 준비해야 합니까?**

실행 파일과 라이브러리를 담는 Docker 이미지, 접근 권한, 시작에 필요한 설정을 작성하고 Colab에서 확인할 파일·설정과 실제 컨테이너·쿠버네티스에서 확인할 동작을 구분해 기록합니다. D66에서 서비스 버전과 실험 결과를 기록한 데 이어 오늘은 컨테이너 실행 상태와 요청 처리 준비 상태를 검사하는 설정을 다룹니다. 먼저 답과 이유를 두 문장으로 적습니다. 실습을 마치면 처음 판단을 자신의 결과로 고칩니다.

**학습목표**

- [목표 1] 이미지를 만드는 빌드 단계와 컨테이너 실행 단계를 구분하고 구성 이유를 설명할 수 있다.
- [목표 2] 사용자 권한·API 키·읽기 전용 설정을 확인해 실행에 필요한 권한을 정할 수 있다.
- [목표 3] 프로세스의 생존 상태와 요청 처리 준비 상태를 검사하고 버전 교체·복구에 필요한 설정을 작성할 수 있다.

**Dockerfile**은 이미지에 넣을 파일과 서버를 시작할 명령을 적은 파일입니다. **쿠버네티스 매니페스트**에는 그 이미지로 몇 개를 실행하고 언제 요청을 받을지 적습니다. 이번에는 작고 필요한 권한만 가진 이미지를 작성한 뒤, 질의응답을 흉내 내는 **mock 서버**로 실제 배포 전에 준비 상태 판단을 연습합니다.

## 이론과 실습의 연계 구성

| 시간 예시 | 분 | 활동 | 실습 노트 위치 |
|---|---:|---|---|
| 10:00~11:05 | 65 | 이미지 고정과 비특권 실행 확인 | Step 0~2 |
| 11:05~11:25 | 20 | 휴식 |  |
| 11:25~13:00 | 95 | 반입 서명 확인과 매니페스트 작성 | Step 3·5 |
| 13:00~14:00 | 60 | 점심 |  |
| 14:00~15:30 | 90 | 같은 서버의 프로브 설정 비교 | Step 6 |
| 15:30~15:50 | 20 | 휴식 |  |
| 15:50~16:40 | 50 | 503과 Ready 상태 해석, 종료와 5분 확인 | Step 9 |
| 16:40~17:00 | 20 | 설치 지연·오류 대응 여유 |  |

**오늘 확인할 질문**　프로세스가 살아 있어도 준비되지 않은 서버에 요청이 들어가면 무엇을 바꿔야 할까?

**비교와 판정**　같은 기동 조건에서 readiness 경로만 바꿔 준비 전 503을 비교합니다. startup 허용 시간은 판정된 기동 시간과 대조해 정합니다.

**7시간 구성**　필수 학습 300분(이론·실습·5분 마무리), 점심 60분, 휴식 40분, 오류 대응 20분입니다. 표에 있는 Step을 순서대로 실행합니다. 오류 대응 시간은 지연이 발생한 구간에 씁니다.

Docker·kind·kubectl이 준비된 팀은 선택 Step 7에서 Colab이 만든 파일을 로컬 터미널로 옮겨 적용·롤링 업데이트·롤백을 확인합니다.

![D67의 입력에서 결과까지 이어지는 학습 흐름](lms-img:0)

**실습 순서.** 실행 이미지 구성 → 권한·비밀값 설정 → 요청 처리 준비 확인 → 버전 교체와 복구 순서로 실습합니다.

### 핵심 용어 정리

| 용어 | 뜻과 실제로 하는 일 |
|---|---|
| 이미지 |  컨테이너 실행에 필요한 프로그램과 라이브러리를 묶은 파일 |
| 파드 |  쿠버네티스에서 함께 실행하는 컨테이너의 묶음 |
| 프로브 |  프로그램의 생존 상태나 요청 처리 준비 상태를 확인하는 검사 |
| 시크릿 |  API 키처럼 외부에 공개하면 안 되는 비밀값 |

## 실행 환경

| 항목 | 내용 |
|---|---|
| 기준 환경 | 코랩 CPU 런타임. GPU가 없어도 모든 셀이 실행됩니다 |
| 코랩에 없는 것 | Docker와 쿠버네티스 클러스터. 이미지 빌드, compose 실행, kind는  로컬 작업입니다 |
| 로드하는 모델 | 없습니다. 서버는 준비 시간을 마음대로 정하는 mock이고 GPU 매니페스트는 Step 8 트랙 1에서 정적 검사만 합니다 |
| 기동하는 서버 | 질의응답 mock 서버 1\~3개(별도 프로세스) |
| 포트 | `8670`부터. 사용 중이면 다음 빈 포트를 씁니다 |
| 오래 걸리는 셀 | Step 0 설치 1\~2분, Step 6 프로브 리허설 두 셀 합쳐 1분 |
|  로컬 | Step 1의 빌드, Step 7의 kind 클러스터 20\~30분. 부록 실습 B의 크기 비교, C의 이미지 검사, F의 compose 실행, I의 클러스터 실습 |
| 컨테이너 노트 파일 | 컨테이너 노트 3.2절의 `Dockerfile`과 5.3절의 `compose.yaml` 내용을 셀 안에 그대로 옮겨 두었습니다 |
| LLM 호출 | 없습니다. 진짜 모델로 바꿀 때는 시스템 프롬프트에 "답변은 반드시 한국어로만 작성하세요"를 넣습니다 |

**로컬 작업** : Docker나 kind를 호출하는 셀은 그 프로그램을 찾지 못하면 로컬에서 입력할 명령을 출력하고 넘어갑니다. 로컬 실습 환경은 컨테이너 노트 2절에서 설치한 Docker(Windows WSL2, macOS, Linux)를 그대로 씁니다. 터미널 명령이 셸마다 다르면 Mac 터미널 버전과 Windows PowerShell 버전을 함께 적었습니다.

## Step 0. 환경 준비와 mock 서버

설치하는 `kubernetes-validate`는 쿠버네티스 설정 파일의 필드와 구조를 점검하는 도구이고, `PyYAML`은 그 설정을 파이썬에서 읽는 라이브러리입니다. Colab에서는 먼저 파일을 점검하고, 뒤에서 mock 서버의 응답으로 준비 상태를 비교합니다.

버전을 고정해 설치합니다. 설치된 버전을 출력하는 셀은 부록 실습 A에 있습니다.

In [2]:
# 패키지 설치. %pip 은 코랩과 로컬 주피터 모두 이 노트북을 실행하는 파이썬에 설치합니다
%pip install -q "kubernetes>=36,<37" "kubernetes-validate>=1.36,<2" "cyclonedx-bom>=7.5,<8" "pyyaml>=6.0,<7" \
                "fastapi>=0.141,<0.142" "uvicorn[standard]>=0.53,<0.54" "pydantic>=2.13,<3" "httpx>=0.28,<1" "anyio>=4.15,<5"
# Dockerfile 린터는 한 줄로 따로 설치합니다. 이 줄이 실패해도 위 패키지는 설치됩니다
%pip install -q "hadolint-bin>=2.15,<3"

In [3]:
import json, os, re, shutil, socket, subprocess, sys, sysconfig, time
from pathlib import Path


def found(cmd, *args):
    return shutil.which(cmd) is not None and subprocess.run([cmd, *args], capture_output=True).returncode == 0


HAS_DOCKER, HAS_KIND, HAS_KUBECTL = found("docker", "info"), found("kind", "version"), found("kubectl", "version", "--client")
print(f"Docker {HAS_DOCKER}  kind {HAS_KIND}  kubectl {HAS_KUBECTL}")
HADOLINT = shutil.which("hadolint") or shutil.which("hadolint", path=sysconfig.get_path("scripts")) or "hadolint"

# 심화 셀은 이 노트북이 실행되는 환경에서 같은 실습 클러스터에 접근할 수 있을 때 사용합니다.
def sh(*cmd, check=True):
    if cmd[0] == "kubectl":
        cmd = ("kubectl", "--context", "kind-d67", *cmd[1:])
    out = subprocess.run(cmd, capture_output=True, text=True)
    if check and out.returncode:
        print("실패:", " ".join(cmd), out.stderr[-300:])
    return out.returncode, out.stdout.strip()


def kind_ready():
    return (HAS_DOCKER and HAS_KIND and HAS_KUBECTL
            and sh("kubectl", "get", "deployment", "qa", check=False)[0] == 0)

Docker False  kind False  kubectl False


`cryptography`는 코랩에 이미 설치되어 있어 설치 목록에 넣지 않았습니다. Step 3에서 서명의 원리를 보일 때만 씁니다.

작업 폴더와 포트를 정하고 mock 서버를 기동하고 끄는 함수를 둡니다. mock 서버가 실제로 답하는 모습은 부록 실습 A에서 자세히 살펴봅니다.

In [4]:
WORK = Path(os.environ.get("D67_WORK", "d67")).resolve()
for d in ["qa", "k8s", "logs", "keys", "out", "tools"]:
    (WORK / d).mkdir(parents=True, exist_ok=True)
(WORK / "qa" / "__init__.py").write_text("", encoding="utf-8")
os.chdir(WORK)
sys.path.insert(0, str(WORK))
BASE_PORT = int(os.environ.get("D67_PORT_BASE", "8670"))
측정 = {}  # 오늘 측정한 수치


def pick_port(taken=()):
    """BASE_PORT 부터 아무도 듣고 있지 않은 포트를 찾습니다."""
    for port in range(BASE_PORT, BASE_PORT + 20):
        with socket.socket() as s:
            if port not in taken and s.connect_ex(("127.0.0.1", port)) != 0:
                return port


def spawn(name, port, env):
    """mock 서버 프로세스 하나를 기동합니다. env 는 D67_ 환경 변수입니다."""
    env = {**os.environ, "D67_NAME": name, "D67_PORT": str(port), **{k: str(v) for k, v in env.items()}}
    log = open(WORK / "logs" / f"{name}.log", "a")
    return subprocess.Popen([sys.executable, "-m", "qa.serve"], cwd=WORK, env=env, stdout=log, stderr=log)


def kill(proc):
    """SIGTERM 을 보내고 5초 안에 끝나지 않으면 강제로 종료합니다."""
    if proc and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            proc.kill()

**mock 서버** : 버전(`D67_VERSION`)과 준비 시간(`D67_WARMUP`)을 환경 변수로 받으므로 파일 하나로 "모델을 로드하는 데 8초 걸리는 서버"를 재현할 수 있습니다.

- `/health` : 프로세스가 답할 수 있으면 언제나 200이고 준비 여부는 본문의 `status`가 `loading`인지 `healthy`인지로 알립니다(게이트웨이 노트 4.1절).  
- `/ready` : 요청을 받을 준비가 됐으면 200, 아니면 503으로 **상태 코드 자체가** 준비 여부를 알립니다. 왜 따로 두는지는 Step 6에서 측정합니다.

In [5]:
%%writefile qa/app.py
# 질의응답 mock 서버. /health 는 언제나 200(준비 여부는 status), /ready 와 /ask 는 준비 전에 503
import asyncio
import os
from contextlib import asynccontextmanager

from fastapi import FastAPI
from fastapi.responses import JSONResponse

VERSION = os.environ.get("D67_VERSION", "1.5.0")
STATE = {"준비됨": False}


@asynccontextmanager
async def lifespan(app):
    async def warm_up():
        await asyncio.sleep(float(os.environ.get("D67_WARMUP", "0")))  # 실제 서버라면 모델을 로드하는 시간
        STATE["준비됨"] = True

    task = asyncio.create_task(warm_up())
    yield
    task.cancel()


app = FastAPI(lifespan=lifespan)
NOT_READY = {"status_code": 503, "content": {"error": "모델을 로드하는 중입니다"}, "headers": {"Retry-After": "1"}}


@app.get("/health")
async def health():
    return {"status": "healthy" if STATE["준비됨"] else "loading", "판": VERSION}


@app.get("/ready")
async def ready():
    return {"ready": True, "판": VERSION} if STATE["준비됨"] else JSONResponse(**NOT_READY)


@app.post("/ask")
async def ask(req: dict):
    if not STATE["준비됨"]:
        return JSONResponse(**NOT_READY)
    await asyncio.sleep(0.05)
    return {"answer": "mock 답", "판": VERSION}

Overwriting qa/app.py


In [6]:
%%writefile qa/serve.py
# 기동 진입점. D67_LOAD_FIRST=1 이면 모델을 다 로드한 뒤에야 포트를 엽니다(vLLM 0.30 처럼)
import os
import time

import uvicorn

if os.environ.get("D67_LOAD_FIRST") == "1":
    time.sleep(float(os.environ.pop("D67_WARMUP", "0")))
uvicorn.run("qa.app:app", host=os.environ.get("D67_HOST", "127.0.0.1"),  # 이미지에만 0.0.0.0 을 적습니다
            port=int(os.environ.get("D67_PORT", "8000")), log_level="warning")

Overwriting qa/serve.py


컨테이너 안에서 실행될 헬스체크도 파일로 둡니다. `python:3.12-slim` 이미지에는 `curl`이 없어서 compose 문서의 예시처럼 `curl -f`를 적으면 헬스체크가 언제나 실패합니다.

In [7]:
%%writefile qa/healthcheck.py
# 컨테이너 안의 헬스체크. 슬림 이미지에는 curl 이 없어 표준 라이브러리만 씁니다. status 가 healthy 일 때만 0
import json
import os
import sys
import urllib.request

try:
    with urllib.request.urlopen(f"http://127.0.0.1:{os.environ.get('D67_PORT', '8000')}/health", timeout=2) as r:
        sys.exit(0 if json.load(r).get("status") == "healthy" else 1)
except OSError as e:  # 연결 실패, 시간 초과
    sys.exit(f"헬스체크 실패 : {type(e).__name__}")

Overwriting qa/healthcheck.py


In [8]:
%%writefile requirements.txt
# 질의응답 서버 이미지에 들어갈 패키지. 2026.10.02 PyPI 확인
fastapi>=0.141,<0.142
uvicorn[standard]>=0.53,<0.54
pydantic>=2.13,<3
anyio>=4.15,<5

Overwriting requirements.txt


In [9]:
# 빌드 컨텍스트에서 뺄 것(컨테이너 노트 4.1절). 키와 반입 파일은 이미지에 넣지 않습니다
Path(".dockerignore").write_text("__pycache__/\n**/.env\nkeys/\nout/\nlogs/\nk8s/\ntools/\n*.tar\n", encoding="utf-8")
import httpx

## Step 1. 멀티스테이지와 숫자 사용자

교재 : 6.1.2 2단계: LLM 컨테이너화하기

우리 서버의 Dockerfile을 빌드 스테이지와 런타임 스테이지로 나누고 root가 아닌 사용자로 실행합니다. 베이스는 태그 대신 **다이제스트**로 고정합니다(이론 노트 3.2절). 다이제스트는 이미지 내용에서 계산한 `sha256` 값이라 내용이 한 바이트만 달라도 바뀝니다. Docker Hub는 태그마다 지금의 다이제스트와 마지막으로 푸시된 때를 알려 주므로 Docker 없이 HTTP 요청 하나로 조회합니다. 이미지 크기를 Docker Hub에서 읽어 비교하는 셀은 부록 실습 B에 있습니다.

**이론 1.1~1.2절 적용**　빌드 도구는 설치물을 만드는 데 쓰고, 배포 이미지에는 실행에 필요한 파일만 복사합니다. 도구까지 배포하면 새 노드가 내려받아야 할 파일이 늘어납니다. Colab에서는 Dockerfile 검사로 실행 스테이지의 사용자와 베이스 식별자를 읽고, 이미지 크기는 실제 Docker에서 빌드한 경우에 비교합니다.

In [10]:
def hub_digest(tag, repo="library/python"):
    """허브에서 태그의 지금 다이제스트를 조회합니다. 실패하면 2026.10.02 에 기록한 값(없으면 None)."""
    try:
        info = httpx.get(f"https://hub.docker.com/v2/repositories/{repo}/tags/{tag}", timeout=10).raise_for_status().json()
        return info["digest"], f"지금 읽음, 마지막 푸시 {str(info.get('tag_last_pushed'))[:10]}"
    except (httpx.HTTPError, ValueError, KeyError):
        return {"3.12-slim": "sha256:dddfd7e07f9d15aeeca61529320492139d21cac7f0070c00609243e51e4e0016"}.get(tag), "기록한 값"


digest, source = hub_digest("3.12-slim")
BASE = f"python:3.12-slim@{digest}"
print(f"베이스 : {BASE}  ({source})")
Path("Dockerfile").write_text(f"""\
ARG BASE_IMAGE={BASE}

# 1. 빌드 스테이지 : 가상환경을 만들고 패키지를 설치합니다
FROM ${{BASE_IMAGE}} AS build
WORKDIR /build
RUN python -m venv --without-pip /opt/venv
COPY requirements.txt .
RUN pip --python /opt/venv/bin/python install --no-cache-dir -r requirements.txt

# 2. 런타임 스테이지 : 가상환경만 받아 오고 숫자 사용자로 실행합니다. 코드는 root 의 것으로 둡니다
FROM ${{BASE_IMAGE}}
RUN groupadd --system --gid 10001 app \\
 && useradd --system --uid 10001 --gid 10001 --no-create-home app
WORKDIR /app
COPY --from=build /opt/venv /opt/venv
COPY qa/ ./qa/
ARG VERSION=1.5.0
ENV PATH=/opt/venv/bin:$PATH D67_VERSION=${{VERSION}} D67_HOST=0.0.0.0 D67_PORT=8000
USER 10001:10001
EXPOSE 8000
CMD ["python", "-m", "qa.serve"]
""", encoding="utf-8")

베이스 : python:3.12-slim@sha256:191083cdf6c7bda8c76158983163ead5d5cf7b77cf3033e805bb0f5296f51b4b  (지금 읽음, 마지막 푸시 2026-11-24)


749

- `FROM ... AS build` : 빌드 스테이지에 이름을 붙였습니다. 이 스테이지에서 만든 파일은 `COPY --from=build`로 옮긴 것만 최종 이미지에 남습니다.  
- `USER 10001:10001` : 사용자 이름이 아니라 숫자로 적었습니다. 이유는 Step 2와 Step 5에서 봅니다.

Dockerfile을 린터 둘로 읽습니다. **hadolint**는 Dockerfile을 읽고 흔한 실수를 규칙 번호와 함께 알려 주는 공용 린터입니다. `latest` 태그(`DL3007`), 다운로드한 패키지 사본을 남기는 `pip install`(`DL3042`), 셸 형식으로 적은 `CMD`(`DL3025`)가 그 예입니다. hadolint는 Dockerfile의 텍스트만 보므로 런타임 스테이지의 다이제스트, 사용자, 실행 형식과 `ENV`·`ARG`에 적은 시크릿 이름을 보는 이 노트의 검사를 하나 더 둡니다. 비교 대상은 컨테이너 노트 3.2절의 `Dockerfile`입니다.

In [11]:
SECRET_NAME = re.compile(r"KEY|SECRET|TOKEN|PASSW|CREDENTIAL", re.I)


def check_stages(path):
    """런타임 스테이지(마지막 FROM 뒤)를 보는 이 노트의 검사. [(규칙, 설명), ...]"""
    text = re.sub(r"\\\n", " ", Path(path).read_text(encoding="utf-8"))
    lines = [l.strip() for l in text.splitlines() if l.strip() and not l.strip().startswith("#")]
    ops = [(l.split(None, 1) + [""])[:2] for l in lines]  # (명령, 나머지)
    args = dict(rest.split("=", 1) for op, rest in ops if op == "ARG" and "=" in rest)
    last = max(i for i, (op, _) in enumerate(ops) if op == "FROM")
    base = re.sub(r"\$\{(\w+)\}", lambda m: args.get(m.group(1), ""), ops[last][1].split()[0])
    users = [rest.split(":")[0] for op, rest in ops[last:] if op == "USER"]
    found = [] if "@sha256:" in base else [("버전", f"{base} : 다이제스트가 없습니다")]
    if not users or users[-1] in ("root", "0"):
        found.append(("root", "런타임 스테이지가 root 로 실행됩니다"))
    elif not users[-1].isdigit():
        found.append(("사용자", f"USER {users[-1]} : 숫자가 아니면 kubelet 이 확인하지 못합니다"))
    if any(op in ("CMD", "ENTRYPOINT") and not rest.startswith("[") for op, rest in ops[last:]):
        found.append(("실행 형식", "셸 형식이라 SIGTERM 이 서버에 바로 전달되지 않습니다"))
    if any(op in ("ENV", "ARG") and SECRET_NAME.search(rest.split("=")[0]) for op, rest in ops):
        found.append(("시크릿", "ENV·ARG 에 키 이름이 있어 이미지 기록에 남습니다"))
    return found


def hadolint(path):
    out = subprocess.run([HADOLINT, "--no-color", str(path)], capture_output=True, text=True)
    return [line.split(" ", 1)[-1] for line in out.stdout.splitlines() if line.strip()]


# 컨테이너 노트 3.2절의 Dockerfile 그대로입니다
Path("out/Dockerfile.d50").write_text("""\
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app/ ./app/
ENV MODEL_VERSION="1.0.0"
EXPOSE 8000
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
""", encoding="utf-8")
for name in ["out/Dockerfile.d50", "Dockerfile"]:
    common, ours = hadolint(name), check_stages(name)
    측정[f"Step 1 {name} 지적 (hadolint, 이 노트)"] = f"{len(common)}건, {len(ours)}건"
    print(f"[{name}]\n  hadolint   {common}\n  이 노트의 검사 {ours}")

[out/Dockerfile.d50]
  hadolint   []
  이 노트의 검사 [('버전', 'python:3.11-slim : 다이제스트가 없습니다'), ('root', '런타임 스테이지가 root 로 실행됩니다')]
[Dockerfile]
  hadolint   []
  이 노트의 검사 []


컨테이너 노트의 `Dockerfile`은 이 노트의 검사에서 다이제스트와 root 두 건이 나옵니다. hadolint의 root 규칙(DL3002)은 마지막 `USER`가 root라고 **적혀 있을 때** 탐지하므로 `USER` 줄이 아예 없는 파일은 지나칩니다. 이 노트의 검사는 `USER`가 없으면 root로 실행된다고 봅니다.

 Docker가 있는 기계에서는 이미지를 실제로 빌드합니다. 크기와 실행 사용자를 확인하고 스테이지를 나누지 않은 버전과 비교하는 셀은 부록 실습 B에 있습니다.

In [12]:
def build(dockerfile, tag, *extra):
    """🖥 이미지를 빌드하고 성공하면 True 를 반환합니다."""
    out = subprocess.run(["docker", "build", "-q", "-f", dockerfile, "-t", tag, *extra, "."], capture_output=True, text=True)
    print(f"{tag} 빌드", "성공" if out.returncode == 0 else "실패 : " + out.stderr[-300:])
    return out.returncode == 0


if HAS_DOCKER:
    build("Dockerfile", "qa-server:1.5.0")
else:
    print("🖥 Docker 가 있는 기계에서 : docker build -t qa-server:1.5.0 .")

🖥 Docker 가 있는 기계에서 : docker build -t qa-server:1.5.0 .


## Step 2. 비root 실행에서 실패하는 지점

root가 아닌 사용자로 바꾸면 root일 때는 조용히 지나가던 일이 멈춥니다. 컨테이너 노트 3.1절의 `app/main.py`는 기동하자마자 `LOG_DIR.mkdir(...)`로 로그 폴더를 만들고 `serving.log`를 엽니다. 컨테이너 노트 4.2절은 그 폴더를 `-v $(pwd)/logs:/app/logs`로 붙였습니다. 코랩에서 root 권한으로 사용자를 바꿔 가며 같은 줄을 실행해 봅니다.

**이론 2.2~2.3절 적용**　비특권 계정으로 바꿀 때는 기록을 쓸 위치도 확인합니다. 로그 폴더에 쓸 권한이 없으면 모델을 로드하기 전에 프로세스가 종료될 수 있습니다. 같은 계정에서 쓰기 위치를 바꾸고 기동 성공과 권한 오류를 대조합니다.

In [13]:
# 컨테이너 노트 app/main.py 의 앞부분과 같은 줄 : 로그 폴더를 만들고 로그 파일을 엽니다
SNIPPET = """
import logging, os
from pathlib import Path
LOG_DIR = Path(os.environ.get("LOG_DIR", "logs"))
LOG_DIR.mkdir(parents=True, exist_ok=True)
logging.basicConfig(handlers=[logging.FileHandler(LOG_DIR / "serving.log")])
print("기동합니다. 로그 :", LOG_DIR.resolve() / "serving.log")
"""


def run_as(uid, folder, **env):
    """folder 에서 uid 사용자로 SNIPPET 을 실행하고 마지막 줄을 반환합니다."""
    who = {} if uid == 0 else {"user": uid, "group": uid}
    out = subprocess.run([sys.executable, "-c", SNIPPET], cwd=folder, capture_output=True, text=True,
                         env={"PATH": os.environ["PATH"], **env}, **who)
    return (out.stdout + out.stderr).strip().splitlines()[-1]


if not hasattr(os, "geteuid") or os.geteuid() != 0:  # 윈도우에는 geteuid 가 없습니다
    print("이 셀은 root 권한이 필요합니다. 코랩에서 실행하세요")
else:
    app_dir, mounted = WORK / "out" / "image_app", WORK / "out" / "mounted_logs"  # 이미지의 /app 과 붙일 폴더
    shutil.rmtree(app_dir, ignore_errors=True)
    app_dir.mkdir(parents=True)
    mounted.mkdir(exist_ok=True)
    os.chown(mounted, 10001, 10001)
    os.chmod(WORK, 0o755)
    print("root,  로그 폴더를 붙이지 않음 :", run_as(0, app_dir))
    shutil.rmtree(app_dir / "logs")
    print("10001, 로그 폴더를 붙이지 않음 :", run_as(10001, app_dir))
    print("10001, 로그 폴더를 붙임        :", run_as(10001, app_dir, LOG_DIR=str(mounted)))

root,  로그 폴더를 붙이지 않음 : 기동합니다. 로그 : /content/d67/out/image_app/logs/serving.log
10001, 로그 폴더를 붙이지 않음 : PermissionError: [Errno 13] Permission denied: 'logs'
10001, 로그 폴더를 붙임        : 기동합니다. 로그 : /content/d67/out/mounted_logs/serving.log


root는 로그 폴더를 붙이지 않았는데도 기동했습니다. 이미지 안이었다면 로그는 컨테이너 안에 쌓였다가 컨테이너를 지울 때 함께 사라집니다. 숫자 사용자는 `PermissionError`로 멈췄고 그 사용자가 쓸 수 있는 폴더를 붙이자 기동했습니다.

**root가 아닌 사용자로 바꿀 때 찾을 지점**

- 서버가 파일을 **쓰는** 곳 : 로그 파일, 기본값을 써 넣는 코드, 캐시 폴더(`HF_HOME`, `~/.cache`). 쓰기가 필요한 폴더만 실행 사용자에게 소유권을 주거나(Step 1 실습의 `chown`) 별도 볼륨을 마운트합니다.

 빌드한 이미지가 실제로 어떤 사용자로 실행되고 코드 폴더에 쓸 수 있는지는 부록 실습 B에서 확인합니다.

## Step 3. 서명한 반입 명세서

교재 : 6.4 자격 증명 관리, 8.2.3 기타 보안 위험

**반입**은 외부에서 가져온 파일을 사내 시스템에 넣는 작업입니다. 서명은 어떤 키로 명세서를 승인했는지 검증하는 데 쓰고, 파일 해시는 반입할 파일이 기록한 내용과 같은지 확인하는 데 씁니다.

**반입 명세서**는 사내망으로 들고 들어가는 파일마다 무엇을 확인했는지 적은 JSON 파일입니다(이론 노트 3.3절). 기본 항목은 `docker save` 파일의 sha256, 크기, 이미지 ID, 이미지 안의 시크릿 검사 결과이고 v2는 베이스 다이제스트, SBOM, 모델의 커밋 해시를 더한 뒤 명세서 전체에 서명합니다.

이 Step은 서명과 확인에 집중합니다.  Docker가 있으면 Step 1의 이미지를 `docker save`로 내보낸 `qa-server.tar`를, 코랩에서는 Dockerfile을 반입물로 씁니다. 컨테이너 이미지 내부의 시크릿 스캔은 부록 실습 C에서, 패키지 목록(SBOM) 생성은 부록 실습 D에서 실행한 뒤 두 검사 결과를 반입 명세서에 추가해 다시 서명합니다.

**이론 3.2~3.3절 적용**　검증한 파일과 다른 파일을 들여오면 측정했던 버전을 재현할 수 없습니다. 공개 키로 명세서의 서명을, 해시로 실제 파일을 대조합니다. 검증 결과에서 원본과 변경된 연습용 파일이 각각 어떻게 판정되는지 봅니다.

서명으로는 **누가 이 명세서를 만들었고 그 뒤로 한 글자도 바뀌지 않았는가**를 확인합니다. 코랩에는 서명 도구 cosign이 없으므로 원리를 같은 방식의 서명(Ed25519)으로 봅니다.

In [14]:
import hashlib
import tarfile

try:
    from cryptography.exceptions import InvalidSignature
    from cryptography.hazmat.primitives import serialization
    from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
    HAS_CRYPTO = True
except ImportError:
    HAS_CRYPTO = False
    print("cryptography 가 없습니다")


def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def canonical(record):  # 키 순서와 띄어쓰기를 고정해야 같은 명세서가 같은 바이트가 됩니다
    return json.dumps(record, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode()


if HAS_CRYPTO:
    SIGNING_KEY = Ed25519PrivateKey.generate()  # 외부망(빌드하는 쪽)만 갖는 키
    Path("keys/release.pub").write_bytes(SIGNING_KEY.public_key().public_bytes(  # 사내망에 미리 반입해 둔 공개 키
        serialization.Encoding.PEM, serialization.PublicFormat.SubjectPublicKeyInfo))

In [15]:
if HAS_DOCKER and not Path("qa-server.tar").exists():  # 🖥 Step 1 의 이미지를 파일 하나로 내보냅니다
    subprocess.run(["docker", "save", "qa-server:1.5.0", "-o", "qa-server.tar"])
artifact = "qa-server.tar" if Path("qa-server.tar").exists() else "Dockerfile"
RECORD = {"반입물": artifact, "파일 sha256": sha256_file(artifact), "크기 MB": round(Path(artifact).stat().st_size / 1e6, 2),
          "베이스 다이제스트": BASE.split("@")[-1], "모델 커밋 해시": "없음 (mock 서버는 가중치를 쓰지 않습니다)",
          "시크릿 검사": "하지 않음 (부록 실습 C)", "SBOM": "없음 (부록 실습 D)", "꺼낸 날": time.strftime("%Y-%m-%d")}
if artifact.endswith(".tar"):
    with tarfile.open(artifact) as tf:
        RECORD["이미지 ID"] = json.load(tf.extractfile("manifest.json"))[0]["Config"].rsplit("/", 1)[-1][:12]
if HAS_CRYPTO:
    SIGNATURE = SIGNING_KEY.sign(canonical(RECORD))
print(json.dumps(RECORD, ensure_ascii=False, indent=1))

{
 "반입물": "Dockerfile",
 "파일 sha256": "a259aa52cb648e18181a7dc477f7ce3cb1a0885d9020db9d96b00c78b3f49ac4",
 "크기 MB": 0.0,
 "베이스 다이제스트": "sha256:191083cdf6c7bda8c76158983163ead5d5cf7b77cf3033e805bb0f5296f51b4b",
 "모델 커밋 해시": "없음 (mock 서버는 가중치를 쓰지 않습니다)",
 "시크릿 검사": "하지 않음 (부록 실습 C)",
 "SBOM": "없음 (부록 실습 D)",
 "꺼낸 날": "2026-11-25"
}


사내망 쪽에서 하는 확인입니다. 전송 중에 파일 한 바이트가 바뀐 경우와 명세서의 해시를 누가 고친 경우를 함께 테스트합니다.

In [16]:
def verify_import(record, signature, artifact, public_pem):
    """서명과 파일 해시가 모두 맞아야 반입합니다."""
    try:
        serialization.load_pem_public_key(public_pem).verify(signature, canonical(record))
    except (InvalidSignature, ValueError):
        return "반입 거절 : 서명이 맞지 않습니다"
    # 서명된 기록을 실제 파일과 대조해 내용이 바뀐 반입물을 걸러냅니다.
    if sha256_file(artifact) != record.get("파일 sha256"):
        return "반입 거절 : 파일이 명세서와 다릅니다"
    return "반입 허용"


if HAS_CRYPTO:
    pub = Path("keys/release.pub").read_bytes()
    tampered = Path("out") / ("tampered-" + Path(artifact).name)
    tampered.write_bytes(Path(artifact).read_bytes() + b"\n")  # 전송 중에 한 바이트가 바뀐 파일
    forged = {**RECORD, "파일 sha256": sha256_file(tampered)}  # 명세서의 해시까지 고친 경우
    for label, rec, art in [("그대로", RECORD, artifact), ("파일이 바뀜", RECORD, tampered),
                            ("명세서를 고침", forged, tampered)]:
        print(f"{label:8} : {verify_import(rec, SIGNATURE, art, pub)}")

그대로      : 반입 허용
파일이 바뀜   : 반입 거절 : 파일이 명세서와 다릅니다
명세서를 고침  : 반입 거절 : 서명이 맞지 않습니다


파일 해시만 적었다면 명세서의 해시를 함께 고친 셋째 경우를 막지 못합니다. 서명은 개인 키의 주인만 만들 수 있으므로 명세서를 고치면 서명이 맞지 않습니다.

## Step 4. compose 헬스체크와 기동 순서 (선택)

Compose에서든 쿠버네티스에서든 서버 기동과 요청을 받을 준비는 구분합니다. 본 실습은 Step 3 뒤 Step 5로 이어집니다.

오늘은 앞선 서비스의 헬스체크가 `healthy`를 반환한 뒤에야 다음 compose 서비스를 기동하도록 `compose.yaml`을 구성합니다. 컨테이너 노트 5.3절에서 쓴 짧은 형식(`depends_on: [api]`)은 API 컨테이너가 시작되기만 하면 바로 화면(`ui`)을 띄우므로 모델 로드가 끝날 때까지 기다려 주지 않습니다.

헬스체크가 **언제** `healthy`가 되는지는 설정 다섯 개가 정합니다. 기본값(`interval` 30초)이면 서버가 8초에 준비를 마쳐도 첫 검사가 30초에 실행되므로 30초를 기다립니다. 시작 기간을 60초로 두고 그 안에서 1초마다 물으면 8초입니다. 대신 시작 기간보다 오래 걸리는 서버는 `unhealthy`가 되므로 **시작 기간은 서버의 기동에 걸리는 가장 긴 시간**으로 잡습니다. Docker 문서의 규칙대로 계산하는 함수와 진짜 서버 프로세스로 확인하는 리허설은 부록 실습 F에 있습니다.

compose 파일을 검사하는 함수를 만듭니다. 헬스체크 없는 compose 서비스를 기다리게 하거나 이미지에 없는 `curl`을 호출하는 실수, 짧은 형식의 `depends_on`을 탐지합니다. 오늘의 mock 서버로 `smoke`가 `qa`의 `healthy`를 기다리게 한 `compose.yaml`은 부록 실습 F에서 쓰고 실제로 기동합니다.

In [17]:
import yaml


def check_compose(spec, no_curl=("qa-server", "python")):
    """compose 의 헬스체크와 기동 순서를 검사합니다. [(서비스, 설명), ...]"""
    found, services = [], spec.get("services", {})
    for name, s in services.items():
        deps = s.get("depends_on") or {}
        if isinstance(deps, list):
            found += [(name, f"{d} 를 짧은 형식으로 기다립니다. 시작만 하면 바로 기동됩니다") for d in deps]
            deps = {}
        for d, cond in deps.items():
            if cond.get("condition") == "service_healthy" and not services.get(d, {}).get("healthcheck"):
                found.append((name, f"{d} 에 healthcheck 가 없어 healthy 가 될 수 없습니다"))
        hc = s.get("healthcheck") or {}
        image = str(s.get("image", "python")).split(":")[0]  # build 만 적었으면 슬림 베이스
        if re.search(r"\b(curl|wget)\b", str(hc.get("test"))) and image in no_curl:
            found.append((name, "슬림 이미지에는 curl 이 없어 헬스체크가 언제나 실패합니다"))
        if hc and "start_period" not in hc:
            found.append((name, "start_period 가 없어 첫 검사가 interval 뒤에야 실행됩니다"))
    return found

#### 실습. 컨테이너 노트의 compose.yaml 고치기

컨테이너 노트 5.3절의 `compose.yaml`에서 화면(`ui`)이 API 서버(`api`)가 요청을 받을 준비를 마친 뒤에 기동되도록 고칩니다. API 서버 이미지에는 `qa/healthcheck.py`가 없으므로 `python -c`로 같은 일을 하는 한 줄을 씁니다. API 서버는 8000번에서 요청을 받고 `/health`의 `status`가 `healthy`이면 준비된 것입니다. Docker가 있는 기계에서 실제로 기동해 기다린 시간을 재는 셀은 부록 실습 F에 있습니다.

In [18]:
# 실습 : 컨테이너 노트 5.3절의 compose.yaml 에서 ui 가 api 의 healthy 를 기다리게 고치기 (GPU 주석 줄은 뺐습니다)
D50_COMPOSE = """\
services:
  api:
    build:
      context: .
      dockerfile: Dockerfile
    ports:
      - "127.0.0.1:8000:8000"
    env_file:
      - .env
    volumes:
      - ./models:/app/models:ro
      - ./logs:/app/logs
  ui:
    build:
      context: .
      dockerfile: Dockerfile.ui
    ports:
      - "127.0.0.1:8501:8501"
    environment:
      - API_BASE_URL=http://api:8000
    depends_on:
      - api
"""
# api 에 붙일 헬스체크. 이미지 안의 파이썬 한 줄로 /health 의 status 를 봅니다
D50_HEALTH = """\
    healthcheck:
      test: ["CMD", "python", "-c", "import json, urllib.request as u; raise SystemExit(json.load(u.urlopen('http://127.0.0.1:8000/health', timeout=2)).get('status') != 'healthy')"]
      start_period: 30s
      start_interval: 1s
"""

### START CODE HERE ### (api 의 마지막 볼륨 줄 뒤에 D50_HEALTH 넣기, ui 의 depends_on 을 긴 형식으로 바꾸기 4줄)
# 1) api 의 마지막 볼륨 줄 바로 뒤에 D50_HEALTH 를 이어 붙입니다
fixed = D50_COMPOSE.replace("      - ./logs:/app/logs\n", "      - ./logs:/app/logs\n" + D50_HEALTH)
# 2) ui 의 짧은 형식(- api)을 condition: service_healthy 의 긴 형식으로 바꿉니다
fixed = fixed.replace("    depends_on:\n      - api\n", "    depends_on:\n      api:\n        condition: service_healthy\n")
### END CODE HERE ###

print("원래 파일 :", check_compose(yaml.safe_load(D50_COMPOSE)))
print("고친 파일 :", check_compose(yaml.safe_load(fixed)) or "지적 없음")
Path("out/compose.d50.yaml").write_text(fixed, encoding="utf-8")

원래 파일 : [('ui', 'api 를 짧은 형식으로 기다립니다. 시작만 하면 바로 기동됩니다')]
고친 파일 : 지적 없음


683

<details class="lms-answer">
<summary>풀이 보기</summary>

**빈칸 정답 코드 (`### START CODE HERE ###` 영역)**

```python
fixed = D50_COMPOSE.replace("      - ./logs:/app/logs\n", "      - ./logs:/app/logs\n" + D50_HEALTH)
fixed = fixed.replace("    depends_on:\n      - api\n", "    depends_on:\n      api:\n        condition: service_healthy\n")
```

`urlopen`은 503 같은 오류 응답에서 예외를 내고 파이썬은 예외로 끝나면 종료 코드 1을 반환하므로 따로 거르지 않아도 실패가 됩니다. `raise SystemExit(False)`는 종료 코드 0, `SystemExit(True)`는 1입니다. 컨테이너 노트의 서버는 가중치를 읽은 뒤에 포트를 여므로 `start_period`는 기동 시간을 재서 정합니다. 고친 파일 `out/compose.d50.yaml`을 컨테이너 노트의 `docker_lab` 폴더에 `compose.yaml`로 두면  `docker compose up -d --wait`로 확인할 수 있습니다.

</details>

## Step 5. 쿠버네티스 매니페스트 작성

교재 : 6.7.2 마이크로서비스 기반 아키텍처 구현, 6.11 다중 모델 오케스트레이션

매니페스트는 쿠버네티스에게 **원하는 상태**를 적어 보내는 YAML 파일입니다. 질의응답 서버의 설정(ConfigMap), 파드 두 개(복제본 수 2)를 기동하고 유지하는 디플로이먼트(Deployment), 두 파드 앞의 고정 주소(Service)를 한 파일에 작성합니다.

온프레미스에서는 외부·경계(DMZ)·업무·모델 서빙 구역을 나누고, 구역 간 연결은 기본 차단합니다. 출발지·목적지·포트·프로토콜과 목적을 적은 허용표에 따라 필요한 연결만 열고, 실제 연결이 예상대로 허용되거나 차단되는지 확인합니다.

매니페스트를 읽는 PyYAML은 환경 준비에서 설치합니다. 아래 검사 셀에서 불러옵니다.

**이론 5.3~5.5절·6.1절 적용**　복제본 수는 실행할 파드 수를, readiness는 그중 새 요청을 받을 수 있는 파드를 정합니다. 복제본이 두 개여도 준비 검사 경로가 잘못되면 로딩 중인 파드가 요청을 받아 503을 돌려줄 수 있습니다. 매니페스트에서 Service가 고르는 파드와 readiness 경로를 확인한 뒤 Step 6에서 모의 요청을 비교합니다.

In [19]:
%%writefile k8s/qa.yaml
apiVersion: v1
kind: ConfigMap
metadata:
  name: qa-config
data:
  D67_WARMUP: "8"
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: qa
spec:
  replicas: 2
  selector:
    matchLabels:
      app: qa
  strategy:
    rollingUpdate:
      maxSurge: 1
      maxUnavailable: 0
  template:
    metadata:
      labels:
        app: qa
    spec:
      terminationGracePeriodSeconds: 30
      securityContext:
        runAsNonRoot: true
        runAsUser: 10001
      containers:
        - name: qa
          image: qa-server:1.5.0
          envFrom:
            - configMapRef:
                name: qa-config
          ports:
            - name: http
              containerPort: 8000
          resources:
            requests:
              cpu: 100m
              memory: 128Mi
            limits:
              memory: 256Mi
          startupProbe:
            httpGet: {path: /health, port: http}
            periodSeconds: 2
            failureThreshold: 30
          readinessProbe:
            # 모델 로딩 중인 파드로 사용자 요청을 보내지 않도록 준비 상태를 검사합니다.
            httpGet: {path: /ready, port: http}
            periodSeconds: 2
          livenessProbe:
            httpGet: {path: /health, port: http}
            periodSeconds: 10
          securityContext:
            allowPrivilegeEscalation: false
            readOnlyRootFilesystem: true
            capabilities:
              drop: ["ALL"]
---
apiVersion: v1
kind: Service
metadata:
  name: qa
spec:
  selector:
    app: qa
  ports:
    - name: http
      port: 8000
      targetPort: http

Overwriting k8s/qa.yaml


- `resources` : CPU와 메모리를 **요청**하고 메모리에는 **상한**을 걸었습니다. CPU 상한은 일부러 두지 않았습니다. 이론 노트 5.5절의 판단입니다.  
- `runAsUser: 10001` : Step 1의 이미지와 같은 숫자입니다. `readOnlyRootFilesystem`은 이 서버가 파일을 쓰지 않으므로 켤 수 있습니다(Step 2). 컨테이너 노트의 서버처럼 로그를 쓰는 서버는 그 폴더만 `emptyDir`로 붙입니다.

게이트웨이를 쿠버네티스로 옮기는 매니페스트는 부록 실습 G에 있습니다. 매니페스트를 두 단계로 검사합니다. 첫 단계는 **규격**입니다. `kubernetes-validate`가 쿠버네티스 1.36의 규격 파일로 필드 이름과 값의 형식을 봅니다.

In [20]:
import yaml

import jsonschema
from kubernetes_validate.utils import schema_contents  # 규격 파일을 버전별로 읽는 도우미
from referencing import Registry, Resource


def schema_errors(doc, version="1.36.0"):
    """한 객체의 쿠버네티스 규격 오류를 모두 반환합니다. [(경로, 설명), ...]"""
    api = re.sub(r"^([^./]*)(?:\.[^/]*)?/", r"\1-", doc["apiVersion"])
    folder = f"kubernetes-json-schema/v{version}-local-strict"
    schema = schema_contents(f"{folder}/{doc['kind'].lower()}-{api}.json", version)
    registry = Resource.from_contents(schema_contents(f"{folder}/_definitions.json", version)) @ (
        Resource.from_contents(schema) @ Registry())
    validator = jsonschema.validators.Draft202012Validator(schema, registry=registry)
    return [(".".join(map(str, e.absolute_path)), e.message) for e in validator.iter_errors(doc)]


QA_DOCS = [d for d in yaml.safe_load_all(Path("k8s/qa.yaml").read_text(encoding="utf-8")) if d]
for doc in QA_DOCS:
    print(f"{doc['kind']:11} {doc['metadata']['name']:12} 규격 오류 {len(schema_errors(doc))}건")

ConfigMap   qa-config    규격 오류 0건
Deployment  qa           규격 오류 0건
Service     qa           규격 오류 0건


둘째 단계는 **우리 규칙**입니다. 규격에는 맞아도 운영에서 사고가 나는 설정을 탐지합니다.

In [21]:
def budget(probe):  # 실패 허용 시간(초). 쿠버네티스 문서의 계산과 같습니다
    return probe.get("initialDelaySeconds", 0) + probe.get("failureThreshold", 3) * probe.get("periodSeconds", 10)


def check_manifests(docs, load_seconds=0, loading_ok_paths=("/health",)):
    """규격 다음 단계의 우리 규칙. 디플로이먼트의 컨테이너마다 걸린 [(규칙, 설명), ...]"""
    found = []
    for pod in [d["spec"]["template"]["spec"] for d in docs if d.get("kind") == "Deployment"]:
        for c in pod["containers"]:
            res, ready = c.get("resources") or {}, c.get("readinessProbe") or {}
            guard = c.get("startupProbe") or c.get("livenessProbe")
            sc = {**(pod.get("securityContext") or {}), **(c.get("securityContext") or {})}
            found += [(rule, why) for rule, bad, why in [
                ("버전", not re.search(r":(?!latest$)[^/:]+$", c["image"].split("@")[0]), "latest 이거나 태그가 없습니다"),
                ("리소스 요청", not {"cpu", "memory"} <= set(res.get("requests") or {}), "배치할 기계를 계산하지 못합니다"),
                ("메모리 상한", "memory" not in (res.get("limits") or {}), "한 파드가 기계의 메모리를 다 쓸 수 있습니다"),
                ("SIGTERM", (c.get("command") or [])[:2] == ["/bin/sh", "-c"], "셸이 SIGTERM 을 먼저 받습니다"),
                ("readiness", not ready or ready.get("httpGet", {}).get("path") in loading_ok_paths,
                 "readiness 프로브가 없거나 로드 중에도 200 을 주는 경로입니다"),
                ("startup", bool(guard and load_seconds and budget(guard) < load_seconds), "기동을 마치기 전에 재시작합니다"),
                ("root", not sc.get("runAsNonRoot"), "root 로 실행될 수 있습니다")] if bad]
    return found


print("k8s/qa.yaml 우리 규칙 :", check_manifests(QA_DOCS, load_seconds=8) or "지적 없음")

k8s/qa.yaml 우리 규칙 : 지적 없음


0건입니다. 동료가 처음 써 온 매니페스트를 두 검사로 읽고 고치는 실습은 부록 실습 J에 있습니다. 오타 하나(`resource`), 따옴표로 감싼 숫자(`"2"`), `latest` 태그, 짧은 startup 프로브, `/health`에 건 readiness 프로브가 어떻게 걸리는지 봅니다.

![규격 검사는 형식의 실수를, 우리 규칙은 형식이 맞아도 운영에서 사고가 나는 설정을 탐지합니다.](lms-img:0)

*규격 검사는 형식의 실수를, 우리 규칙은 형식이 맞아도 운영에서 사고가 나는 설정을 탐지합니다.*

## Step 6. 프로브 리허설

교재 : 6.11 다중 모델 오케스트레이션

**프로브(probe)**는 서버가 시작됐는지, 새 요청을 받을 준비가 됐는지, 재시작이 필요한지를 자동으로 확인하는 검사입니다. 먼저 준비 상태(readiness)를 틀리게 판단했을 때 사용자 요청이 어떤 응답을 받는지 비교합니다.

매니페스트의 프로브가 실제로 무슨 일을 하는지 코랩에서 재현합니다. kubelet의 규칙을 그대로 따르는 작은 감시 코드를 만들어 진짜 서버 프로세스에 붙입니다.

**kubelet의 규칙** (쿠버네티스 1.36 문서와 규격)

- HTTP 프로브는 **상태 코드가 200 이상 400 미만이면 성공**입니다. 본문은 읽지 않습니다.  
- startup 프로브(`startupProbe`)를 설정하면 startup 프로브가 성공할 때까지 liveness·readiness 프로브는 실행되지 않습니다. startup 프로브가 연속 `failureThreshold`번 실패하면 컨테이너를 재시작합니다.  
- liveness 프로브(`livenessProbe`)가 연속 `failureThreshold`번 실패하면 컨테이너를 재시작합니다.  
- readiness 프로브(`readinessProbe`)가 성공해야 Service가 그 파드에 요청을 보냅니다.

시간을 줄이려고 리허설에서는 간격을 1초로 둡니다.

In [22]:
import itertools
import threading


def probe_once(port, path):  # 상태 코드가 200 이상 400 미만이면 성공. 본문은 읽지 않습니다
    try:
        return 200 <= httpx.get(f"http://127.0.0.1:{port}{path}", timeout=1).status_code < 400
    except httpx.HTTPError:
        return False


class Pod:
    """컨테이너 하나를 kubelet 의 규칙대로 1초마다 감시합니다(모의). 프로브는 (경로, 연속 실패 허용 횟수)."""

    def __init__(self, name, env, startup=None, liveness=None, readiness=None, start_after=0):
        self.name, self.env, self.start_after = name, env, start_after
        self.specs = {"startup": startup, "liveness": liveness, "readiness": readiness}
        self.restarts, self.first_ready, self.proc, self.port, self.ready = 0, None, None, None, False

    def start(self):
        self.port = pick_port(taken={p.port for p in PODS if p.proc})
        self.proc = spawn(self.name, self.port, self.env)
        self.started, self.ready = self.specs["startup"] is None, False  # startup 프로브가 없으면 바로 다음 단계
        self.fails = dict.fromkeys(self.specs, 0)

    def tick(self, now):
        for kind, spec in self.specs.items():
            if spec is None or (kind == "startup") == self.started:  # startup 이 성공해야 나머지가 실행됩니다
                continue
            ok = probe_once(self.port, spec[0])
            self.fails[kind] = 0 if ok else self.fails[kind] + 1
            self.started = self.started or (kind == "startup" and ok)
            if kind == "readiness":
                self.ready = ok or (self.ready and self.fails[kind] < spec[1])
                self.first_ready = self.first_ready or (now if ok else None)
            elif self.fails[kind] >= spec[1]:  # kubelet 처럼 재시작합니다
                kill(self.proc)
                self.restarts += 1
                self.start()
                return self.tick(now)  # 새 컨테이너에도 곧바로 묻습니다


PODS = []


def run_pods(pods, seconds, traffic=False):
    """seconds 초 동안 감시합니다. traffic 이면 Service 처럼 Ready 상태의 파드에만 0.2초마다 요청합니다."""
    PODS[:] = pods
    t0, tally, rr, stop = time.time(), {}, itertools.count(), threading.Event()

    def watch():
        while True:
            for p in pods:
                if p.proc is None and time.time() - t0 >= p.start_after:
                    p.start()
                if p.proc:
                    p.tick(time.time())  # 기동하자마자 첫 검사, 그 뒤 1초마다
            if stop.wait(1.0):
                return

    threading.Thread(target=watch, daemon=True).start()
    t_next = t0
    while time.time() - t0 < seconds:
        ready = [p for p in pods if p.ready]
        if traffic and (ready or tally):  # 첫 파드가 준비된 뒤부터 요청합니다
            target = ready[next(rr) % len(ready)] if ready else None
            try:
                key = httpx.post(f"http://127.0.0.1:{target.port}/ask", json={"question": "q"}).status_code if target else "보낼 곳 없음"
            except httpx.HTTPError as e:
                key = type(e).__name__
            tally[key] = tally.get(key, 0) + 1
        t_next += 0.2
        time.sleep(max(0.0, t_next - time.time()))
    stop.set()
    for p in pods:
        kill(p.proc)
    return tally, {p.name: (p.restarts, p.first_ready and round(p.first_ready - t0 - p.start_after, 1)) for p in pods}

**이론 6.1~6.2절 적용**　같은 모델 기동 조건에서 readiness 검사 경로만 `/health`와 `/ready`로 바꿉니다. 프로브가 응답 본문을 읽지 않으므로 로딩 중에도 200을 주는 `/health`를 쓰면 준비 전 트래픽이 들어갈 수 있습니다. 기존 파드는 계속 실행하고, 새 파드는 2초 뒤 기동해 모델을 4초 동안 읽습니다. 실행 전에 어느 조건에서 503이 나올지 예상합니다.

In [23]:
# 다른 기동 조건은 유지하고 검사 경로만 바꿔 준비 전 503의 원인을 확인합니다.
for path in ["/health", "/ready"]:
    old = Pod("old-1", {"D67_WARMUP": 0}, readiness=(path, 3))
    new = Pod("new-1", {"D67_WARMUP": 4}, readiness=(path, 3), start_after=2)
    tally, pods = run_pods([old, new], 8, traffic=True)
    print(f"readiness = {path:8} 응답 {tally}   새 파드가 Ready 로 판정된 때 : 기동 뒤 {pods['new-1'][1]}초")
    측정[f"Step 6 readiness {path} 의 503"] = f"{tally.get(503, 0)}건 / {sum(tally.values())}건"

readiness = /health  응답 {200: 25, 503: 9}   새 파드가 Ready 로 판정된 때 : 기동 뒤 1.3초
readiness = /ready   응답 {200: 33}   새 파드가 Ready 로 판정된 때 : 기동 뒤 5.8초


**출력에서 확인**　각 경로의 `새 파드가 Ready로 판정된 때`와 503 건수를 대조합니다. 이 리허설에서 `/health`로 검사하면 모델 로드 전에 Ready가 되고, `/ready`로 검사하면 준비가 끝나기 전에는 새 파드가 트래픽을 받지 않습니다. 어떤 경로를 매니페스트에 쓸지 두 출력으로 설명합니다.

![kubelet은 상태 코드만 읽으므로 readiness 프로브는 준비 전에 503을 반환하는 경로에 걸어야 합니다.](lms-img:0)

*kubelet은 상태 코드만 읽으므로 readiness 프로브는 준비 전에 503을 반환하는 경로에 걸어야 합니다.*

**이론 6.3절 적용**　이번에는 포트를 열기까지 6초가 걸리는 서버를 씁니다. liveness는 1초 간격으로 3회 실패하면 재시작하도록 두고, startup만 없음·5초·15초로 바꿉니다. 초기화가 끝나기 전에 재시작하면 서버가 준비 상태에 도달하지 못합니다. 각 설정의 재시작 횟수와 처음 준비된 시점을 예상합니다.

In [24]:
# liveness가 초기화를 끊는지 보기 위해 startup의 허용 구간만 바꿉니다.
for label, startup in [("startup 프로브 없음", None), ("startup 프로브 5초", ("/health", 5)), ("startup 프로브 15초", ("/health", 15))]:
    pod = Pod("llm-1", {"D67_WARMUP": 6, "D67_LOAD_FIRST": 1}, startup, liveness=("/health", 3), readiness=("/ready", 3))
    restarts, first = run_pods([pod], 12)[1]["llm-1"]
    print(f"{label:14} 12초 동안 재시작 {restarts}번, 처음 준비 :", f"{first}초" if first else "없음")
    측정[f"Step 6 {label} : 재시작"] = f"{restarts}번"

startup 프로브 없음 12초 동안 재시작 5번, 처음 준비 : 없음
startup 프로브 5초 12초 동안 재시작 2번, 처음 준비 : 없음
startup 프로브 15초 12초 동안 재시작 0번, 처음 준비 : 7.4초


**출력에서 확인**　기동 시간 6초와 각 설정의 재시작 횟수·처음 준비 시점을 대조합니다. 설정값을 실제 서비스에 옮길 때는 대상 장비에서 모델 로드 시간을 재고 여유를 둡니다. 모의 실행 결과는 어떤 검사에서 문제가 생기는지 설명하는 근거로 씁니다.

**롤링 업데이트에서 빈 GPU가 없을 때** : GPU를 한 장씩 쓰는 서버를 빈 GPU 없이 기본값(25%)으로 롤링하면 서버가 세 대 이하일 때 `maxSurge`는 1, `maxUnavailable`은 0이 됩니다. 새 파드는 빈 GPU가 없어 배치되지 못하고 옛 파드는 하나도 종료할 수 없어 롤링이 멈춥니다. `maxSurge: 0`, `maxUnavailable: 1`로 두면 옛 파드 하나를 먼저 종료해 비운 GPU에 새 파드를 기동하므로 끝까지 가고 바꾸는 동안 요청을 받는 서버가 한 대 줄어듭니다(이론 노트 6.4절). 스텝마다 계산하는 계획표와 실습은 부록 실습 H에 있습니다.

## Step 7. 로컬 터미널에서 쿠버네티스 배포 (선택)

**kind**는 Docker 컨테이너를 노드로 사용하는 로컬 쿠버네티스입니다. **kubectl**은 터미널에서 쿠버네티스에 설정을 보내고 실제 상태를 읽는 명령입니다. Step 5에서 만든 매니페스트를 적용하고, Step 6에서 배운 준비 상태가 롤링 업데이트 중 실제 파드에 어떻게 나타나는지 봅니다.

Docker가 실행 중이고 kind와 kubectl이 준비된 팀은 약 20~25분을 잡습니다. 터미널에서 `docker info`, `kind version`, `kubectl version --client`가 모두 성공해야 합니다. 첫 이미지 다운로드는 사전에 확인합니다. 실습 이미지는 로컬에 넣으므로 별도의 레지스트리 계정이나 토큰이 필요 없습니다. 환경이 준비되지 않은 팀은 Step 6의 리허설 결과로 판단을 남기고 Step 9로 이동합니다. [kind 시작하기](https://kind.sigs.k8s.io/docs/user/quick-start/)

아래 셀은 실습에 필요한 파일만 묶습니다. ZIP을 로컬에 내려받아 압축을 풀고, 터미널에서 `Dockerfile`이 보이는 폴더로 이동합니다.

In [25]:
from zipfile import ZIP_DEFLATED, ZipFile

# Colab에서 만든 배포 파일만 옮깁니다. 키, 로그, 반입 자료는 포함하지 않습니다.
export_files = ("Dockerfile", ".dockerignore", "requirements.txt", "qa/__init__.py",
                "qa/app.py", "qa/serve.py", "k8s/qa.yaml")
missing = [name for name in export_files if not (WORK / name).is_file()]
if missing:
    raise FileNotFoundError(f"Step 0·1·5를 먼저 실행하세요. 누락: {', '.join(missing)}")

archive = WORK / "out" / "d67-kind.zip"
with ZipFile(archive, "w", compression=ZIP_DEFLATED) as bundle:
    for name in export_files:
        bundle.write(WORK / name, arcname=name)
print("배포 파일:", archive)

try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(str(archive))

배포 파일: /content/d67/out/d67-kind.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
from google.colab import drive
drive.mount('/content/drive')
!cp /content/d67/out/d67-kind.zip /content/drive/MyDrive/


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#### 1. 로컬 터미널에서 첫 버전 적용

이름이 `d67`인 클러스터가 이미 있으면 그 소유와 용도를 먼저 확인합니다. 실습용 클러스터를 새로 만들고, 이미지 두 개를 레지스트리에 올리지 않고 노드에 직접 넣습니다. 매니페스트를 적용할 때마다 `--context kind-d67`로 대상 클러스터를 지정합니다. 명령이 실패하면 출력된 원인을 확인한 뒤 다음 명령으로 넘어갑니다.

```powershell
kind create cluster --name d67
docker build -t qa-server:1.5.0 --build-arg VERSION=1.5.0 .
docker build -t qa-server:1.6.0 --build-arg VERSION=1.6.0 .
kind load docker-image qa-server:1.5.0 qa-server:1.6.0 --name d67
kubectl --context kind-d67 apply -f k8s/qa.yaml
kubectl --context kind-d67 rollout status deployment/qa --timeout=120s
kubectl --context kind-d67 get pods -l app=qa
```

처음에 두 파드가 `1/1`인지 확인합니다. `ImagePullBackOff`가 보이면 로컬 이미지 태그가 매니페스트와 같은지, 이미지를 지정한 kind 클러스터에 넣었는지 확인합니다.

#### 2. 다른 터미널에서 가용 파드 수를 보며 업데이트

두 번째 터미널에서 아래 명령으로 디플로이먼트 상태를 지켜봅니다. `AVAILABLE` 열은 요청을 받을 수 있는 파드 수입니다.

```powershell
kubectl --context kind-d67 get deployment qa --watch
```

첫 번째 터미널에서 버전을 올린 뒤, 상태를 확인하고 이전 버전으로 되돌립니다. 롤백이 끝나면 두 번째 터미널의 관찰을 Ctrl+C로 종료합니다.

```powershell
kubectl --context kind-d67 set image deployment/qa qa=qa-server:1.6.0
kubectl --context kind-d67 rollout status deployment/qa --timeout=120s
kubectl --context kind-d67 rollout undo deployment/qa
kubectl --context kind-d67 rollout status deployment/qa --timeout=120s
kubectl --context kind-d67 get deployment qa -o "jsonpath={.spec.template.spec.containers[0].image}"
```

명령이 시간 안에 끝나지 않으면 다른 버전을 연달아 적용하기 전에 아래 출력에서 실패 원인을 확인합니다.

```powershell
kubectl --context kind-d67 get pods -l app=qa
kubectl --context kind-d67 get events --sort-by=.lastTimestamp
```

**이론 5절과 Step 6 적용**　`maxSurge: 1`과 `maxUnavailable: 0`으로 새 파드가 준비될 때까지 기존 파드를 유지합니다. 준비되지 않은 파드를 너무 일찍 활성화하면 사용자 요청이 503을 받을 수 있습니다. 관찰한 최저 `AVAILABLE` 수와 롤백 뒤 이미지 태그를 적고, 화면에서 전환을 놓쳤다면 최저값을 미확인으로 남깁니다. 부록 실습 I는 같은 클러스터에 접근할 수 있는 노트북에서 이어갑니다.

## Step 8. 선택 심화 트랙

시간이 남으면 두 트랙 중 하나를 고릅니다. 트랙 1은 코랩에서 실행할 수 있습니다. kind 상태를 보는 트랙은 선택 Step 7을 먼저 실행합니다.

| 트랙 | 주제 | 성격 | 난이도 |
|---|---|---|---|
| 1 | vLLM 쿠버네티스 배포 매니페스트 | 설계 | 중 |
| 2 |  메모리 상한과 OOMKilled | 측정 | 중 |

다른 주제(이미지 레이어 분석, 서명 키 교체, GPU 부족 재현)는 부록 실습에 있습니다.

#### 트랙 1. vLLM 쿠버네티스 배포 매니페스트
교재 : 6.11 다중 모델 오케스트레이션, 9.5.2 자체 보유 하드웨어에서의 프로비저닝과 관리

**vLLM 0.30 서버는 모델 가중치 적재와 초기화가 모두 끝난 뒤에야 HTTP 포트를 열어 연결을 수락하므로 쿠버네티스 매니페스트에서 `/health` 엔드포인트를 `readinessProbe` 경로로 그대로 사용할 수 있습니다.** 이번 트랙에서는 vLLM 공식 문서(v0.30.0)의 쿠버네티스 배포 예시를 Step 5의 정적 검사기로 점검해 사내 보안·안정성 기준에 맞게 수정하며 기동 유예 시간은 vLLM 기초 노트 4.3절의 T4 실측 기록(147초)을 기준치로 활용합니다.

In [27]:
VLLM_DOC_EXAMPLE = """\
apiVersion: apps/v1
kind: Deployment
metadata:
  name: mistral-7b
spec:
  replicas: 1
  selector:
    matchLabels:
      app: mistral-7b
  template:
    metadata:
      labels:
        app: mistral-7b
    spec:
      containers:
        - name: mistral-7b
          image: vllm/vllm-openai:latest
          command: ["/bin/sh", "-c"]
          args: ["vllm serve mistralai/Mistral-7B-Instruct-v0.3 --trust-remote-code"]
          ports:
            - containerPort: 8000
          resources:
            limits: {cpu: "10", memory: 20G, nvidia.com/gpu: "1"}
            requests: {cpu: "2", memory: 6G, nvidia.com/gpu: "1"}
          livenessProbe:
            httpGet: {path: /health, port: 8000}
            initialDelaySeconds: 60
            periodSeconds: 10
          readinessProbe:
            httpGet: {path: /health, port: 8000}
            initialDelaySeconds: 60
            periodSeconds: 5
"""


def vllm_report(doc):
    rolling = doc["spec"].get("strategy", {}).get("rollingUpdate", {})
    print("규격 오류", len(schema_errors(doc)), "건 | 우리 규칙", check_manifests([doc], 147, loading_ok_paths=()),
          "| 롤링 설정", rolling or "기본값 25% (빈 GPU 0장이면 멈춤)")


vllm_report(yaml.safe_load(VLLM_DOC_EXAMPLE))

규격 오류 0 건 | 우리 규칙 [('버전', 'latest 이거나 태그가 없습니다'), ('SIGTERM', '셸이 SIGTERM 을 먼저 받습니다'), ('startup', '기동을 마치기 전에 재시작합니다'), ('root', 'root 로 실행될 수 있습니다')] | 롤링 설정 기본값 25% (빈 GPU 0장이면 멈춤)


**과제** : `k8s/vllm.yaml`을 작성합니다. 모델은 Qwen2.5-1.5B-Instruct, T4 노드이므로 `--dtype half`, 가중치는 이미지 밖의 볼륨(`PersistentVolumeClaim`)에서 읽습니다. 지적이 남는다면 고치지 못하는 이유를 주석으로 적습니다.

In [28]:
# 실습 : 트랙 1 사내 기준에 맞춘 vLLM 매니페스트 k8s/vllm.yaml 쓰기 (문서 예시에서 고칠 곳만 고칩니다)
### START CODE HERE ### (빈 GPU 없이 끝나는 롤링 설정, startup 프로브의 간격과 실패 횟수 2줄)
SURGE, UNAVAILABLE = 0, 1  # 1) 빈 GPU 가 없을 때 롤링이 끝나는 값 (이론 노트 6.4절)
STARTUP_PERIOD, STARTUP_FAILURES = 10, 30  # 2) 곱한 실패 허용 시간이 기동 시간 어림값 147초보다 넉넉하게
### END CODE HERE ###
doc = yaml.safe_load(VLLM_DOC_EXAMPLE.replace("mistral-7b", "vllm-qa"))
doc["spec"]["strategy"] = {"rollingUpdate": {"maxSurge": SURGE, "maxUnavailable": UNAVAILABLE}}
pod, c = doc["spec"]["template"]["spec"], doc["spec"]["template"]["spec"]["containers"][0]
c.pop("command")  # 이미지의 ENTRYPOINT 가 이미 vllm serve 라 셸로 감싸지 않습니다
c["image"] = "vllm/vllm-openai:v0.30.0@sha256:5f5e535216848d0c52159c8c13a0af04be5f6fe1a84e79914300610796f76d40"
c["args"] = ["--model", "Qwen/Qwen2.5-1.5B-Instruct", "--dtype", "half", "--max-model-len", "4096"]
c["env"] = [{"name": "HF_HOME", "value": "/models/hf"}]  # 가중치는 이미지 밖의 볼륨에서 읽습니다
c["startupProbe"] = {"httpGet": {"path": "/health", "port": 8000},
                     "periodSeconds": STARTUP_PERIOD, "failureThreshold": STARTUP_FAILURES}
for probe in ["livenessProbe", "readinessProbe"]:
    c[probe].pop("initialDelaySeconds")  # 기동을 기다리는 일은 startup 프로브가 맡습니다
c["volumeMounts"] = [{"name": "models", "mountPath": "/models"}, {"name": "shm", "mountPath": "/dev/shm"}]
pod["volumes"] = [{"name": "models", "persistentVolumeClaim": {"claimName": "models"}},
                  {"name": "shm", "emptyDir": {"medium": "Memory", "sizeLimit": "2Gi"}}]
Path("k8s/vllm.yaml").write_text(yaml.safe_dump(doc, sort_keys=False), encoding="utf-8")
vllm_report(doc)

규격 오류 0 건 | 우리 규칙 [('root', 'root 로 실행될 수 있습니다')] | 롤링 설정 {'maxSurge': 0, 'maxUnavailable': 1}


<details class="lms-answer">
<summary>풀이 보기</summary>

**빈칸 정답 코드 (`### START CODE HERE ###` 영역)**

```python
SURGE, UNAVAILABLE = 0, 1
STARTUP_PERIOD, STARTUP_FAILURES = 10, 30
```

10초 × 30번이면 실패 허용 시간이 300초로 147초의 두 배 남짓입니다. 남는 지적은 root 하나입니다. 공식 `vllm/vllm-openai` 이미지는 root로 실행됩니다(이론 노트 2.3절의 `vllm-openai-nonroot` 스테이지로 직접 빌드하거나 예외로 승인받습니다).

</details>

#### 트랙 2.  메모리 상한과 OOMKilled

메모리 상한을 넘은 컨테이너는 커널이 강제 종료하고 쿠버네티스는 그 사유를 `OOMKilled`로 남깁니다. CPU 상한을 넘으면 종료하지 않고 느리게 만듭니다. Step 7의 클러스터에서 상한 128Mi인 파드가 300MB를 할당하게 해 봅니다.

In [29]:
OOM_POD = """\
apiVersion: v1
kind: Pod
metadata:
  name: oom-test
spec:
  restartPolicy: Never
  securityContext: {runAsNonRoot: true, runAsUser: 10001}
  containers:
    - name: hog
      image: qa-server:1.5.0
      command: ["python", "-c", "x = bytearray(300 * 1024 * 1024); import time; time.sleep(30)"]
      resources: {requests: {memory: 64Mi}, limits: {memory: 128Mi}}
"""
Path("k8s/oom.yaml").write_text(OOM_POD, encoding="utf-8")
print("규격 오류 :", schema_errors(yaml.safe_load(OOM_POD)))
if not kind_ready():
    print("🖥 Step 7의 실습 클러스터에 접근할 수 있는 로컬 노트북에서 실행하세요.")
else:
    sh("kubectl", "apply", "-f", "k8s/oom.yaml")
    time.sleep(20)
    print("종료 사유 :", sh("kubectl", "get", "pod", "oom-test", "-o",
                          "jsonpath={.status.containerStatuses[0].state.terminated.reason}")[1] or "(아직 종료되지 않음)")
    sh("kubectl", "delete", "pod", "oom-test")

규격 오류 : []
🖥 Step 7의 실습 클러스터에 접근할 수 있는 로컬 노트북에서 실행하세요.


**과제** : 상한을 512Mi로 올려 같은 파드가 끝까지 실행되는지 확인합니다. 그다음 LLM 서버의 메모리 상한을 정할 때 무엇을 측정해야 하는지 적습니다.

## Step 9. 공유와 정리

노트북에서 띄운 서버를 정리하고 오늘 측정한 수치를 봅니다. 로컬 kind까지 진행한 팀은 관찰한 가용 파드 수와 롤백 뒤 버전을 함께 남깁니다.

In [30]:
for p in PODS:
    kill(p.proc)
for name, value in 측정.items():
    print(f"  {name:44} {value}")

  Step 1 out/Dockerfile.d50 지적 (hadolint, 이 노트) 0건, 2건
  Step 1 Dockerfile 지적 (hadolint, 이 노트)        0건, 0건
  Step 6 readiness /health 의 503               9건 / 34건
  Step 6 readiness /ready 의 503                0건 / 33건
  Step 6 startup 프로브 없음 : 재시작                  5번
  Step 6 startup 프로브 5초 : 재시작                  2번
  Step 6 startup 프로브 15초 : 재시작                 0번


In [31]:
print("Step 7에서 직접 만든 실습 클러스터는 부록 I까지 마친 뒤, 로컬 터미널에서 kind delete cluster --name d67로 정리합니다.")

Step 7에서 직접 만든 실습 클러스터는 부록 I까지 마친 뒤, 로컬 터미널에서 kind delete cluster --name d67로 정리합니다.


readiness를 바꿨을 때 준비 전 503이 어떻게 달라졌는지 보고, 측정한 기동 시간으로 startup 허용 시간을 고릅니다.

### 5분 마무리 확인

프로세스는 살아 있지만 요청을 받을 준비가 되지 않았습니다. 어떤 프로브를 보고 재시작과 새 요청 차단 중 무엇을 선택할지 설명합니다. 답은 팀에서 말로 공유하거나 기존 실습 기록에 한 줄로 남깁니다.

| 자기 점검 | 남길 근거 |
|---|---|
| [목표 1] 바꾸고 실행한 내용 | 사용한 입력과 설정, 실제 출력 |
| [목표 2] 비교한 결과와 판단 | 그대로 둔 설정과 바꾼 설정, 직접 측정한 결과인지 여부 |
| [목표 3] 다시 실행할 때 필요한 기록 | 저장한 파일과 선택에 사용한 측정값 |

D68에서는 준비한 작업을 실행 순서대로 연결하고 실패한 작업을 다시 실행할 규칙을 정합니다.

## 부록 실습 A. 설치한 버전과 mock 서버 확인

본 실습을 마친 뒤 시간이 남거나 더 깊이 보고 싶을 때 하는 실습입니다. 본 실습 Step 9까지 실행한 상태에서 시작합니다(부록 실습 I는 Step 7의 클러스터를 지우기 전에 합니다).

실제로 설치된 버전을 출력해 둡니다. 결과가 달라졌을 때 가장 먼저 볼 기록입니다.

In [32]:
from importlib.metadata import PackageNotFoundError, version

for name in ["kubernetes", "kubernetes-validate", "cyclonedx-bom", "hadolint-bin",
             "pyyaml", "fastapi", "uvicorn", "pydantic", "httpx", "anyio", "cryptography"]:
    try:
        print(f"{name:20} {version(name)}")
    except PackageNotFoundError:
        print(f"{name:20} (설치되지 않음)")
print("파이썬               ", sys.version.split()[0])

kubernetes           36.0.3
kubernetes-validate  1.37.0
cyclonedx-bom        7.6.0
hadolint-bin         2.15.1
pyyaml               6.0.3
fastapi              0.141.1
uvicorn              0.53.0
pydantic             2.14.2
httpx                0.28.1
anyio                4.16.0
cryptography         50.0.2
파이썬                3.13.18


Step 0의 mock 서버를 준비 2초로 기동해 `/health`와 `/ready`가 어떻게 답하는지 봅니다.

In [33]:
port = pick_port()
proc = spawn("try-1", port, {"D67_WARMUP": 2})
for sec in range(1, 5):
    time.sleep(1)
    try:
        print(f"{sec}초  /health", httpx.get(f"http://127.0.0.1:{port}/health").json()["status"],
              " /ready", httpx.get(f"http://127.0.0.1:{port}/ready").status_code)
    except httpx.HTTPError as e:
        print(f"{sec}초  아직 연결되지 않음 ({type(e).__name__})")
kill(proc)

1초  /health loading  /ready 503
2초  /health loading  /ready 503
3초  /health healthy  /ready 200
4초  /health healthy  /ready 200


모델을 로드하는 2초 동안 `/health`는 200에 `loading`을 반환하고 `/ready`는 503을 반환합니다. 2초가 지나면 둘 다 200입니다. 1초 줄이 "아직 연결되지 않음"이면 프로세스가 기동되는 데 1초가 넘게 걸렸습니다.

## 부록 실습 B. 이미지 크기와 레이어

Docker Hub API에서 GPU 베이스 이미지 세 종류와 vLLM 공식 이미지의 크기를 읽습니다(이론 노트 1장 심화 ‘엔비디아 CUDA 베이스 이미지의 세 가지 유형(base·runtime·devel)’, 3).

In [34]:
# 2026.10.02 Docker Hub API 에서 읽은 값 (amd64, 압축 크기). 허브에 접속할 수 없을 때만 씁니다
RECORDED_SIZES = {
    "python:3.12-slim": 46_220_071,
    "nvidia/cuda:13.0.3-base-ubuntu24.04": 141_958_546,
    "nvidia/cuda:13.0.3-runtime-ubuntu24.04": 1_650_371_769,
    "nvidia/cuda:13.0.3-devel-ubuntu24.04": 3_973_716_026,
    "vllm/vllm-openai:v0.30.0": 8_730_525_875,
}


def hub_size(image):
    """허브에서 amd64 이미지의 압축 크기를 읽습니다. 실패하면 기록한 값을 반환합니다."""
    repo, tag = image.split(":")
    repo = repo if "/" in repo else "library/" + repo
    try:
        r = httpx.get(f"https://hub.docker.com/v2/repositories/{repo}/tags/{tag}", timeout=10)
        r.raise_for_status()
        for img in r.json().get("images", []):
            if img.get("architecture") == "amd64":
                return img["size"], "지금 읽음"
    except (httpx.HTTPError, ValueError, KeyError):
        pass
    return RECORDED_SIZES.get(image), "기록한 값 (2026.10.02)"


print(f"{'이미지':42} {'압축 크기':>10}  출처")
for image in RECORDED_SIZES:
    size, source = hub_size(image)
    shown = "알 수 없음" if size is None else f"{size / 1e9:7.2f} GB"
    print(f"{image:42} {shown:>10}  {source}")

이미지                                             압축 크기  출처
python:3.12-slim                              0.04 GB  지금 읽음
nvidia/cuda:13.0.3-base-ubuntu24.04           0.14 GB  지금 읽음
nvidia/cuda:13.0.3-runtime-ubuntu24.04        1.65 GB  지금 읽음
nvidia/cuda:13.0.3-devel-ubuntu24.04          3.97 GB  지금 읽음
vllm/vllm-openai:v0.30.0                      8.73 GB  지금 읽음


- CUDA 이미지 셋은 같은 CUDA 13.0.3입니다. `runtime`은 `base`보다 1.5GB쯤 크고 `devel`은 다시 2.3GB쯤 큽니다. `devel`에 더 든 것은 CUDA 코드를 컴파일하는 도구와 헤더입니다.  
- vLLM 공식 이미지는 8.7GB입니다. 이미지 용량의 가장 큰 비중은 pip로 설치하는 PyTorch와 CUDA 관련 라이브러리가 차지합니다. 아래에서 레이어마다 측정합니다.

 스테이지를 나누지 않은 우리 서버 이미지(v1)를 빌드해 Step 1의 이미지와 크기를 비교합니다. 패키지가 전부 wheel 파일이라 크기가 거의 같을 것입니다(이론 노트 1장 심화 ‘멀티스테이지 빌드에서도 이미지 크기가 줄지 않는 원인’).

In [35]:
Path("Dockerfile.v1").write_text(f"""\
# 질의응답 서버 이미지 v1 : 스테이지 하나, USER 없음
FROM {BASE}
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY qa/ ./qa/
ENV D67_VERSION=1.5.0 D67_HOST=0.0.0.0 D67_PORT=8000
EXPOSE 8000
CMD ["python", "-m", "qa.serve"]
""", encoding="utf-8")
print("이 노트의 검사 :", check_stages("Dockerfile.v1"))
if not HAS_DOCKER:
    print("🖥 docker build -f Dockerfile.v1 -t qa-server:1.5.0-v1 .  ;  docker image ls qa-server")
elif build("Dockerfile.v1", "qa-server:1.5.0-v1"):
    for tag in ["qa-server:1.5.0-v1", "qa-server:1.5.0"]:
        size, *user = subprocess.run(["docker", "image", "inspect", "--format", "{{.Size}} {{.Config.User}}", tag],
                                     capture_output=True, text=True).stdout.split() or ["0"]
        print(f"{tag:20} 풀어 놓은 크기 {int(size) / 1e6:.0f} MB  USER {user or '(없음 = root)'}")

이 노트의 검사 : [('root', '런타임 스테이지가 root 로 실행됩니다')]
🖥 docker build -f Dockerfile.v1 -t qa-server:1.5.0-v1 .  ;  docker image ls qa-server


**공식 vLLM 이미지의 레이어 분석**

Docker Hub API는 이미지의 레이어마다 크기와 그 레이어를 만든 명령을 알려 줍니다. vLLM v0.30.0 이미지의 레이어를 읽어 크기가 어디서 오는지 유형별로 계산합니다.

In [36]:
# 2026.10.02 에 읽은 vllm/vllm-openai:v0.30.0 amd64 의 큰 레이어 (크기, 그 레이어를 만든 명령의 유형)
RECORDED_LAYERS = [
    (30_593_586 + 4_547_272 + 105_166_716, "베이스 (nvidia/cuda base)"),
    (233_434_151, "운영체제 패키지 (apt)"),
    (1_453_065_768, "CUDA 개발 도구 (apt, 실행 중 컴파일용)"),
    (41_410_487, "파이썬 패키지 (pip, uv)"),
    (5_113_006_184, "파이썬 패키지 (pip, vLLM과 PyTorch)"),
    (1_016_496_432, "파이썬 패키지 (pip)"),
]
RECORDED_TOTAL = 8_730_525_875


def classify(instruction):
    text = instruction.lower()
    if "cuda-nvcc" in text or "cuda_version_dash" in text:
        return "CUDA 개발 도구 (apt, 실행 중 컴파일용)"
    if "pip install" in text or "uv pip" in text:
        return "파이썬 패키지 (pip)"
    if "apt-get" in text:
        return "운영체제 패키지 (apt)"
    return "그 밖"


def vllm_layers(tag="v0.30.0"):
    try:
        r = httpx.get(f"https://hub.docker.com/v2/repositories/vllm/vllm-openai/tags/{tag}/images", timeout=15)
        r.raise_for_status()
        image = next(i for i in r.json() if i.get("architecture") == "amd64")
        layers = [(l.get("size", 0), classify(l.get("instruction", ""))) for l in image["layers"]]
        return layers, sum(s for s, _ in layers), "지금 읽음"
    except (httpx.HTTPError, ValueError, KeyError, StopIteration):
        rest = RECORDED_TOTAL - sum(s for s, _ in RECORDED_LAYERS)
        return RECORDED_LAYERS + [(rest, "그 밖")], RECORDED_TOTAL, "기록한 값 (2026.10.02)"


layers, total, source = vllm_layers()
groups = {}
for size, label in layers:
    group = "파이썬 패키지 (pip)" if label.startswith("파이썬") else label    # pip 레이어는 하나로 묶습니다
    groups[group] = groups.get(group, 0) + size
print(f"전체 {total / 1e9:.2f} GB ({source}), 가장 큰 레이어 {max(s for s, _ in layers) / 1e9:.2f} GB")
for label, size in sorted(groups.items(), key=lambda kv: -kv[1]):
    print(f"  {label:32} {size / 1e9:6.2f} GB  {size / total:5.1%}")

전체 8.73 GB (지금 읽음), 가장 큰 레이어 5.11 GB
  파이썬 패키지 (pip)                      6.56 GB  75.1%
  CUDA 개발 도구 (apt, 실행 중 컴파일용)        1.80 GB  20.6%
  운영체제 패키지 (apt)                     0.34 GB   3.9%
  그 밖                                0.03 GB   0.4%


**"컴파일러는 빌드 스테이지에만 두고 런타임 이미지에서는 제거한다"는 일반적인 멀티스테이지 빌드 원칙도 런타임 중에 GPU 커널을 JIT 컴파일하는 추론 서버에서는 예외가 됩니다.** 레이어 내역에서 특히 눈여겨볼 항목은 CUDA 개발 도구로, vLLM은 런타임 스테이지를 가벼운 `base` 이미지에서 시작하면서도 실행 시점 커널 컴파일에 필요한 `cuda-nvcc`와 헤더 패키지를 런타임 스테이지에 별도로 설치합니다.

**과제** : 같은 셀을 `tag="v0.29.0"`처럼 다른 버전으로 실행해 크기와 유형별 구성이 어떻게 바뀌었는지 적습니다.

빌드한 이미지를 기동해 실제로 어떤 사용자로 실행되는지와 코드 폴더에 쓸 수 있는지 확인합니다(본 실습 Step 2).

In [37]:
if not HAS_DOCKER:
    print("🖥 docker run --rm qa-server:1.5.0 id  ;  docker run --rm qa-server:1.5.0 touch /app/qa/x   # 쓰기 거부")
else:
    for cmd in [["id"], ["touch", "/app/qa/x"]]:
        out = subprocess.run(["docker", "run", "--rm", "qa-server:1.5.0", *cmd], capture_output=True, text=True)
        print(*cmd, "->", (out.stdout + out.stderr).strip()[-90:])

🖥 docker run --rm qa-server:1.5.0 id  ;  docker run --rm qa-server:1.5.0 touch /app/qa/x   # 쓰기 거부


## 부록 실습 C. 이미지 안의 시크릿 스캐너

교재 : 6.4 자격 증명 관리

컨테이너 노트 4.1절에서 본 시크릿 유입 경로 셋(통째 복사, `ENV`·`ARG`, `RUN rm`)을 `docker save` 파일에서 직접 찾는 스캐너를 만듭니다(이론 노트 2.5절). 표준 라이브러리만 쓰는 파일 하나로 두어 사내망의 어느 기계에서나 실행되게 합니다.

In [38]:
%%writefile tools/scan_image.py
"""이미지 안의 시크릿 스캐너. 표준 라이브러리만 쓰므로 파이썬만 있으면 어느 기계에서나 실행됩니다.

사용 : python tools/scan_image.py qa-server.tar    (docker save 로 내보낸 파일)
탐지된 것이 있으면 종료 코드 1 을 반환합니다. 스캐너도 키 원문을 화면에 다 출력하지 않습니다.
"""
import json
import re
import sys
import tarfile

# 앞에 다른 글자가 붙었거나 중간에 - 가 끼는 sk- 는 키로 보지 않습니다 (첫 버전의 오탐에서 좁힌 규칙)
KEY_TEXT = re.compile(rb"(?<![\w-])(?:sk-[A-Za-z0-9_]{20,}|hf_[A-Za-z0-9]{30,}|AKIA[0-9A-Z]{16})"
                      rb"|-----BEGIN [A-Z ]*PRIVATE KEY-----\s+[A-Za-z0-9+/]{40}")
KEY_NAME = re.compile(r"KEY|SECRET|TOKEN|PASSW|CREDENTIAL", re.I)
SUSPECT_FILE = re.compile(r"(^|/)(\.env|api_keys\.json|[^/]*키[^/]*\.txt)$")
BINARY = (".so", ".pyc", ".png", ".jpg", ".jpeg", ".gz", ".zip", ".whl", ".joblib",
          ".safetensors", ".bin", ".pt", ".onnx")
READ_LIMIT = 256 * 1024


def mask(text):
    """화면에 출력하기 전에 키 원문을 마스킹합니다."""
    return KEY_TEXT.sub(b"(key)", text.encode()).decode()


def judge(name, raw):
    """파일 이름과 내용으로 사유를 반환합니다. 문제가 없으면 None 입니다."""
    if raw is not None and KEY_TEXT.search(raw):
        return "키 원문"
    if SUSPECT_FILE.search(name):
        return "시크릿 의심"
    return None


def readable(name, size):
    """내용까지 읽을 파일인지. 큰 파일과 바이너리는 이름만 봅니다."""
    return size <= READ_LIMIT and not name.lower().endswith(BINARY)


def scan_layer(fileobj):
    """레이어 하나(tar)를 처음부터 끝까지 한 번 읽습니다. 압축 여부는 자동으로 판별합니다."""
    count, hits, removed = 0, [], []
    with tarfile.open(fileobj=fileobj, mode="r|*") as tf:
        for m in tf:
            base = m.name.rsplit("/", 1)[-1]
            if base.startswith(".wh."):
                removed.append(m.name.replace(".wh.", "", 1))
                continue
            if not m.isfile():
                continue
            count += 1
            raw = tf.extractfile(m).read() if readable(m.name, m.size) else None
            reason = judge(m.name, raw)
            if reason:
                hits.append((m.name, reason))
    return count, hits, removed


def scan_image(tar_path):
    """레이어마다 파일을 검사하고 이미지 설정의 Env 와 빌드 기록도 확인합니다."""
    layers, settings = [], []
    with tarfile.open(tar_path) as outer:
        manifest = json.load(outer.extractfile("manifest.json"))[0]
        config = json.load(outer.extractfile(manifest["Config"]))
        made_by = [mask(h.get("created_by") or "(베이스 이미지)")
                   for h in config.get("history", []) if not h.get("empty_layer")]
        for i, name in enumerate(manifest["Layers"]):
            count, hits, removed = scan_layer(outer.extractfile(name))
            label = made_by[i] if i < len(made_by) else "?"
            layers.append((i + 1, label, count, hits, removed))
        for item in config.get("config", {}).get("Env") or []:
            key, _, value = item.partition("=")
            named = KEY_NAME.search(key) and not key.endswith(("FILE", "PATH"))
            if named or KEY_TEXT.search(value.encode()):
                settings.append(f"설정의 Env   {key}={value[:6]}...")
        for h in config.get("history", []):
            text = h.get("created_by") or ""
            if KEY_TEXT.search(text.encode()):
                settings.append("빌드 기록    " + mask(text)[:66])
    return layers, settings


def count_hits(tar_path):
    """탐지된 것의 수. 반입 명세서의 시크릿 검사 항목에 씁니다."""
    layers, settings = scan_image(tar_path)
    return sum(len(hits) for _, _, _, hits, _ in layers) + len(settings)


def main(tar_path):
    """검사 결과를 출력하고 탐지된 것의 수를 반환합니다."""
    layers, settings = scan_image(tar_path)
    for i, label, count, hits, removed in layers:
        print(f"레이어 {i}  파일 {count:5}개  {label[:58]}")
        for name, reason in hits:
            print(f"        {reason:8} {name}")
        for name in removed:
            print(f"        삭제 표시 {name}  <- 아래 레이어에는 그대로 있습니다")
    for line in settings:
        print("  " + line)
    total = sum(len(hits) for _, _, _, hits, _ in layers) + len(settings)
    print(f"탐지된 것 {total}건")
    return total


if __name__ == "__main__":
    if len(sys.argv) != 2:
        print(__doc__)
        sys.exit(2)
    sys.exit(1 if main(sys.argv[1]) else 0)

Writing tools/scan_image.py


연습 이미지로 스캐너부터 테스트합니다. 시크릿이 유입되는 세 경로를 모두 거친 이미지를 `docker save`가 만드는 형식 그대로 재현합니다.

In [39]:
import hashlib
import importlib
import io
import secrets
import tarfile

sys.path.insert(0, str(WORK / "tools"))
import scan_image

importlib.reload(scan_image)


def add_bytes(tf, name, data):
    info = tarfile.TarInfo(name)
    info.size = len(data)
    tf.addfile(info, io.BytesIO(data))


def layer_tar(entries):
    """(이름, 바이트) 목록으로 레이어 하나를 만듭니다."""
    buf = io.BytesIO()
    with tarfile.open(fileobj=buf, mode="w") as tf:
        for name, data in entries:
            add_bytes(tf, name, data)
    return buf.getvalue()


def practice_image(out_path, key):
    """세 경로(통째 복사, ENV·ARG, RUN rm)를 모두 거친 이미지를 docker save 가 만드는 형식 그대로 재현합니다."""
    code = [(f"app/qa/{p.name}", p.read_bytes()) for p in sorted((WORK / "qa").glob("*.py"))]
    copy_layer = layer_tar(code + [("app/.env", f"D67_API_KEY={key}\n".encode())])   # COPY . .
    rm_layer = layer_tar([("app/.wh..env", b"")])                                     # RUN rm -f .env
    config = {"config": {"Env": [f"D67_API_KEY={key}", "D67_PORT=8000"]},
              "history": [
                  {"created_by": f"ARG D67_API_KEY={key}", "empty_layer": True},
                  {"created_by": f"ENV D67_API_KEY={key}", "empty_layer": True},
                  {"created_by": "COPY . . # buildkit"},
                  {"created_by": f"RUN |1 D67_API_KEY={key} /bin/sh -c rm -f .env # buildkit"}]}
    blobs = {}                                       # docker save 처럼 내용의 sha256 을 이름으로 씁니다
    for data in [json.dumps(config).encode(), copy_layer, rm_layer]:
        blobs[f"blobs/sha256/{hashlib.sha256(data).hexdigest()}"] = data
    names = list(blobs)
    manifest = [{"Config": names[0], "RepoTags": ["qa-leaky:practice"], "Layers": names[1:]}]
    with tarfile.open(out_path, mode="w") as tf:
        add_bytes(tf, "manifest.json", json.dumps(manifest).encode())
        for name, data in blobs.items():
            add_bytes(tf, name, data)
    return out_path


LEAKY = practice_image(WORK / "out" / "qa-leaky-practice.tar", "sk-" + secrets.token_hex(12))
found = scan_image.main(LEAKY)
print(f"\n명령줄에서 실행했다면 종료 코드 {1 if found else 0} (1 이면 탐지된 것이 있다는 뜻)")

레이어 1  파일     5개  COPY . . # buildkit
        키 원문     app/.env
레이어 2  파일     0개  RUN |1 D67_API_KEY=(key) /bin/sh -c rm -f .env # buildkit
        삭제 표시 app/.env  <- 아래 레이어에는 그대로 있습니다
  설정의 Env   D67_API_KEY=sk-43d...
  빌드 기록    ARG D67_API_KEY=(key)
  빌드 기록    ENV D67_API_KEY=(key)
  빌드 기록    RUN |1 D67_API_KEY=(key) /bin/sh -c rm -f .env # buildkit
탐지된 것 5건

명령줄에서 실행했다면 종료 코드 1 (1 이면 탐지된 것이 있다는 뜻)


스캐너가 세 경로를 모두 탐지했습니다. 둘째 레이어에는 삭제 표시만 있고 첫째 레이어에 `.env`가 그대로 있다는 것도 따로 알려 줍니다.

 Docker가 있는 기계에서는 Step 1에서 빌드한 `qa-server:1.5.0`을 `docker save`로 내보내 같은 검사를 합니다. `COPY`로 `qa/` 폴더만 고르고 `.dockerignore`가 키와 반입 파일을 빼므로 탐지된 것이 0건이어야 합니다. 스캔 결과를 반입 명세서에 기록하고 다시 서명합니다.

In [40]:
if not HAS_DOCKER:
    print("🖥 Docker 가 있는 기계에서 이 셀을 실행하거나 터미널에서 이렇게 하세요 (Mac 터미널과 Windows PowerShell 에서 같습니다)")
    print("   docker save qa-server:1.5.0 -o qa-server.tar")
    print("   python tools/scan_image.py qa-server.tar")
elif not Path("qa-server.tar").exists():
    print("qa-server.tar 가 없습니다. Step 1 의 빌드와 Step 3 의 첫 셀을 먼저 실행하세요")
else:
    hits = scan_image.main("qa-server.tar")
    측정["부록 C qa-server 시크릿 검사"] = f"{hits}건"
    RECORD["시크릿 검사"] = "통과" if hits == 0 else f"{hits}건 탐지됨. 반입하지 않습니다"
    if HAS_CRYPTO and artifact == "qa-server.tar":
        SIGNATURE = SIGNING_KEY.sign(canonical(RECORD))          # 항목을 채웠으므로 다시 서명합니다
        print(verify_import(RECORD, SIGNATURE, artifact, Path("keys/release.pub").read_bytes()))

🖥 Docker 가 있는 기계에서 이 셀을 실행하거나 터미널에서 이렇게 하세요 (Mac 터미널과 Windows PowerShell 에서 같습니다)
   docker save qa-server:1.5.0 -o qa-server.tar
   python tools/scan_image.py qa-server.tar


## 부록 실습 D. 잠금 파일과 SBOM

`requirements.txt`에 적은 패키지들이 하위 의존성을 포함해 실제로 몇 개가 설치되는지 확인하고 전체 설치 패키지 목록을 표준 SBOM 형식으로 변환합니다(이론 노트 3장 심화 ‘컨테이너 내부 패키지 구성을 기록하는 SBOM’).

In [41]:
lock = subprocess.run([sys.executable, "-m", "pip", "install", "--dry-run", "--ignore-installed",
                       "--quiet", "--report", "out/pip-report.json", "-r", "requirements.txt"],
                      capture_output=True, text=True, encoding="utf-8", errors="replace",
                      env={**os.environ, "PYTHONUTF8": "1"})     # 윈도우에서도 pip 이 한글 주석을 UTF-8 로 읽게 합니다
if lock.returncode != 0:
    print("의존성 해석에 실패했습니다 :", lock.stderr.strip()[-300:])
else:
    report = json.loads(Path("out/pip-report.json").read_text(encoding="utf-8"))
    pins = sorted((f"{i['metadata']['name']}=={i['metadata']['version']}" for i in report["install"]),
                  key=str.lower)
    Path("requirements.lock").write_text(
        "# requirements.txt 에서 고른 버전. 이미지와 같은 파이썬에서 만듭니다\n" + "\n".join(pins) + "\n",
        encoding="utf-8")
    direct = [l for l in Path("requirements.txt").read_text(encoding="utf-8").splitlines() if l and not l.startswith("#")]
    print(f"적은 줄 {len(direct)}개 -> 실제로 들어갈 패키지 {len(pins)}개 "
          f"(파이썬 {report['environment']['python_full_version']})")
    print("  ", ", ".join(pins[:6]), "...")
    측정["부록 D 적은 줄 -> 들어갈 패키지"] = f"{len(direct)} -> {len(pins)}"

적은 줄 4개 -> 실제로 들어갈 패키지 19개 (파이썬 3.13.18)
   annotated-doc==0.0.7, annotated-types==0.8.0, anyio==4.16.0, click==8.5.1, fastapi==0.141.1, h11==0.16.0 ...


`requirements.txt`의 패키지 네 개를 설치하면 하위 의존성을 포함해 열아홉 개 남짓의 패키지가 설치됩니다(파이썬 버전에 따라 한두 개 다릅니다). 잠금 파일은 이미지와 같은 파이썬에서 만들어야 같은 목록이 됩니다.

잠금 파일을 표준 형식의 SBOM으로 바꿉니다.

In [42]:
import sysconfig

CYCLONEDX = shutil.which("cyclonedx-py") or shutil.which("cyclonedx-py", path=sysconfig.get_path("scripts")) or "cyclonedx-py"
sbom = subprocess.run([CYCLONEDX, "requirements", "requirements.lock", "--of", "JSON",
                       "-o", "out/sbom.cdx.json"], capture_output=True, text=True,
                      encoding="utf-8", errors="replace", env={**os.environ, "PYTHONUTF8": "1"})
if sbom.returncode != 0:
    print("SBOM 을 만들지 못했습니다 :", sbom.stderr.strip()[-300:])
else:
    SBOM = json.loads(Path("out/sbom.cdx.json").read_text(encoding="utf-8"))
    print(SBOM["bomFormat"], SBOM["specVersion"], "구성 요소", len(SBOM["components"]), "개")
    for c in SBOM["components"][:3]:
        print("  ", c["name"], c["version"], c["purl"])
    RECORD["SBOM"] = {"sha256": sha256_file("out/sbom.cdx.json"), "구성 요소": len(SBOM["components"])}
    if HAS_CRYPTO:
        SIGNATURE = SIGNING_KEY.sign(canonical(RECORD))          # 항목을 채웠으므로 다시 서명합니다
        print(verify_import(RECORD, SIGNATURE, artifact, Path("keys/release.pub").read_bytes()))

CycloneDX 1.6 구성 요소 19 개
   PyYAML 6.0.4 pkg:pypi/pyyaml@6.0.4
   annotated-doc 0.0.7 pkg:pypi/annotated-doc@0.0.7
   annotated-types 0.8.0 pkg:pypi/annotated-types@0.8.0
반입 허용


## 부록 실습 E. 서명 키 교체

교재 : 6.4 자격 증명 관리

사내망 쪽에는 **믿는 공개 키 목록**을 두고 목록에 든 키 가운데 하나로 서명이 맞으면 반입합니다. 바꾸는 동안에는 옛 키와 새 키를 함께 두고 옛 키를 뺀 뒤에는 옛 키로 서명한 반입물을 반입하지 않습니다.

In [43]:
# 실습 : 부록 실습 E 믿는 공개 키 목록으로 서명 키 교체
def verify_with_any(record, signature, artifact, trusted):
    """trusted : {키 이름: 공개 키 PEM}. 하나라도 맞으면 그 키 이름을 반환합니다. 맞는 키가 없으면 None 입니다."""
    if sha256_file(artifact) != record.get("파일 sha256"):
        return None
    for name, pem in trusted.items():
        try:
            serialization.load_pem_public_key(pem).verify(signature, canonical(record))
            return name
        except (InvalidSignature, ValueError):
            continue
    return None


if HAS_CRYPTO:
    NEW_KEY = Ed25519PrivateKey.generate()
    new_pem = NEW_KEY.public_key().public_bytes(serialization.Encoding.PEM,
                                                serialization.PublicFormat.SubjectPublicKeyInfo)
    old_pem = Path("keys/release.pub").read_bytes()
    new_sig = NEW_KEY.sign(canonical(RECORD))

    ### START CODE HERE ### (바꾸는 동안의 믿는 키 목록과 옛 키를 뺀 뒤의 목록 2줄)
    # 1) 바꾸는 동안 : 옛 키와 새 키를 함께 둡니다 ({키 이름: 공개 키 PEM})
    during = {"2026-old": old_pem, "2026-new": new_pem}
    # 2) 옛 키를 뺀 뒤 : 새 키만 둡니다
    after = {"2026-new": new_pem}
    ### END CODE HERE ###

    for label, TRUSTED in [("바꾸는 동안", during), ("옛 키를 뺀 뒤", after)]:
        if TRUSTED is None:
            print("TRUSTED 가 아직 비어 있습니다. 예 : {'2026-old': old_pem, '2026-new': new_pem}")
        else:
            print(label, "| 옛 키 서명 :", verify_with_any(RECORD, SIGNATURE, artifact, TRUSTED),
                  "| 새 키 서명 :", verify_with_any(RECORD, new_sig, artifact, TRUSTED))

바꾸는 동안 | 옛 키 서명 : 2026-old | 새 키 서명 : 2026-new
옛 키를 뺀 뒤 | 옛 키 서명 : None | 새 키 서명 : 2026-new


<details class="lms-answer">
<summary>풀이 보기</summary>

**빈칸 정답 코드 (`### START CODE HERE ###` 영역)**

```python
during = {"2026-old": old_pem, "2026-new": new_pem}
after = {"2026-new": new_pem}
```

바꾸는 동안에는 두 서명이 모두 통과하고 옛 키를 뺀 뒤에는 새 키로 서명한 것만 통과합니다.

</details>

## 부록 실습 F. compose 헬스체크 계산, 리허설, 실제 기동

헬스체크가 언제 `healthy`가 되는지 Docker 문서의 규칙대로 계산하는 함수부터 만듭니다.

In [44]:
def healthy_at(ready_s, interval=30, retries=3, start_period=0, start_interval=5):
    """ready_s 초에 준비가 끝나는 서버를 compose 가 몇 초에 healthy 로 보는지 계산합니다."""
    t, fails = 0, 0
    while fails < retries:
        t += start_interval if t < start_period else interval
        if t >= ready_s:
            return "healthy", t
        fails += t > start_period                # 시작 기간 안의 실패는 세지 않습니다
    return "unhealthy", t


TUNED = dict(interval=10, retries=3, start_period=60, start_interval=1)
print("준비 8초 서버, 기본값    ->", healthy_at(8))
print("준비 8초 서버, 조정한 값 ->", healthy_at(8, **TUNED))
print("준비 100초 서버, 조정한 값 ->", healthy_at(100, **TUNED))

준비 8초 서버, 기본값    -> ('healthy', 30)
준비 8초 서버, 조정한 값 -> ('healthy', 8)
준비 100초 서버, 조정한 값 -> ('unhealthy', 90)


같은 계산을 진짜 서버 프로세스로 확인합니다. mock 서버 둘을 준비 8초로 기동하고 `qa/healthcheck.py`를 Docker의 규칙대로 실행합니다. 기본값 쪽 때문에 이 셀은 35초쯤 걸립니다.

In [45]:
def rehearse_healthcheck(name, port, interval, start_period=0, start_interval=5, warmup=8):
    """서버를 기동하고 헬스체크 명령을 Docker 의 간격대로 실행해 healthy 가 된 초를 반환합니다."""
    proc = spawn(name, port, {"D67_WARMUP": warmup})
    env = {**os.environ, "D67_PORT": str(port)}
    t0, t = time.time(), 0.0
    try:
        while t < 60:
            t += start_interval if t < start_period else interval
            time.sleep(max(0.0, t0 + t - time.time()))
            if subprocess.run([sys.executable, "-m", "qa.healthcheck"], env=env, cwd=WORK,
                              capture_output=True).returncode == 0:
                return round(time.time() - t0, 1)
        return None
    finally:
        kill(proc)


# 두 서버를 동시에 기동하므로 포트를 미리 따로 잡아 둡니다. 같은 포트를 잡으면 한쪽이 기동되지 못합니다
port_a = pick_port()
port_b = pick_port(taken={port_a})
results = {}
jobs = [threading.Thread(target=lambda: results.update(기본값=rehearse_healthcheck("hc-default", port_a, 30))),
        threading.Thread(target=lambda: results.update(조정한값=rehearse_healthcheck("hc-tuned", port_b, 10, 60, 1)))]
for j in jobs:
    j.start()
for j in jobs:
    j.join()
for label, sec in results.items():
    shown = "60초 안에 healthy 가 되지 못함" if sec is None else f"{sec}초"
    print(f"{label:5} healthy 까지 {shown}")
    측정[f"부록 F 헬스체크 {label} (준비 8초 서버)"] = shown

조정한값  healthy 까지 10.1초
기본값   healthy 까지 30.1초


계산값과 실측값의 차이가 1초 남짓입니다.

본 실습의 mock 서버로 compose 파일을 씁니다. 서버가 `healthy`가 된 뒤에만 질문 하나를 보내고 끝나는 점검 `smoke`가 기동되도록 `depends_on`을 긴 형식으로 적고 Step 4의 검사로 읽습니다.

In [46]:
%%writefile compose.yaml
# 질의응답 서버 하나와 질문 하나를 보내고 끝나는 점검 하나
name: d67
services:
  qa:
    build: .
    image: qa-server:1.5.0
    environment:
      D67_WARMUP: ${D67_WARMUP:-8}
    ports:
      - "127.0.0.1:${D67_QA_PORT:-8670}:8000"
    healthcheck:
      test: ["CMD", "python", "-m", "qa.healthcheck"]
      interval: 10s
      timeout: 3s
      retries: 3
      start_period: 60s
      start_interval: 1s
  smoke:
    image: qa-server:1.5.0
    command: ["python", "-c", "import urllib.request as u; r = u.urlopen(u.Request('http://qa:8000/ask', b'{\"question\": \"q\"}', {'Content-Type': 'application/json'})); print('점검 결과 :', r.status)"]
    depends_on:
      qa:
        condition: service_healthy

Writing compose.yaml


In [47]:
print("compose.yaml :", check_compose(yaml.safe_load(Path("compose.yaml").read_text(encoding="utf-8"))) or "지적 없음")

compose.yaml : 지적 없음


Docker가 있는 기계에서 실제로 기동합니다. 먼저 조정한 compose 파일로 `--wait`이 끝나기까지의 시간과 `smoke`의 결과를 보고 짧은 형식으로 바꾼 파일과 비교합니다.

In [48]:
def compose(*args, file="compose.yaml"):
    out = subprocess.run(["docker", "compose", "-f", file, *args], capture_output=True, text=True,
                         encoding="utf-8", errors="replace")
    return out.returncode, (out.stdout + out.stderr).strip()


if not HAS_DOCKER:
    print("🖥 Docker 가 있는 기계에서 :")
    print("   time docker compose up -d --wait qa                  # Mac·리눅스 터미널")
    print("   Measure-Command { docker compose up -d --wait qa }   # Windows PowerShell")
    print("   docker compose up smoke ; docker compose down")
else:
    started = Path("compose.started.yaml")
    started.write_text(Path("compose.yaml").read_text(encoding="utf-8").replace(
        "    depends_on:\n      qa:\n        condition: service_healthy\n", "    depends_on:\n      - qa\n"),
        encoding="utf-8")
    for file in ["compose.yaml", str(started)]:
        t0 = time.time()
        if file == "compose.yaml":
            compose("up", "-d", "--wait", "qa", file=file)
        else:
            compose("up", "-d", "qa", file=file)
        waited = time.time() - t0
        code, out = compose("up", "smoke", file=file)
        last = [l for l in out.splitlines() if "점검" in l]
        print(f"{file:22} qa 를 기다린 시간 {waited:5.1f}초   smoke : {last[-1] if last else out[-120:]}")
        compose("down", "-t", "5", file=file)

🖥 Docker 가 있는 기계에서 :
   time docker compose up -d --wait qa                  # Mac·리눅스 터미널
   Measure-Command { docker compose up -d --wait qa }   # Windows PowerShell
   docker compose up smoke ; docker compose down


조정한 compose 파일은 `qa`가 `healthy`가 된 뒤에 `smoke`를 기동해 200을 받았습니다. 짧은 형식은 `qa`가 시작되자마자 `smoke`를 기동했고 모델을 로드하던 `qa`가 503을 반환해 `smoke`가 1로 끝났습니다.

## 부록 실습 G. 게이트웨이 매니페스트

게이트웨이 노트 6.1절의 `MiniAPIGateway`를 서버로 감싼 이미지 `registry.example.com/ai/gateway:1.0.0`이 있다고 가정하고 쿠버네티스로 옮깁니다. 질의응답 서버는 Step 5의 디플로이먼트가 맡으므로 게이트웨이의 라우팅 테이블에는 서버 주소 대신 Service 이름 하나만 적습니다.

In [49]:
%%writefile k8s/gateway.yaml
# 게이트웨이. 라우팅 테이블에는 서버 주소 대신 Service 이름 하나만 적습니다
apiVersion: v1
kind: ConfigMap
metadata:
  name: gateway-routes
data:
  routes.json: |
    {"판": "k-001",
     "groups": {"1.5.0": ["http://qa:8000"]},
     "routes": {"/v1/": {"1.5.0": 100}}}
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: gateway
spec:
  replicas: 1
  selector:
    matchLabels:
      app: gateway
  template:
    metadata:
      labels:
        app: gateway
    spec:
      securityContext:
        runAsNonRoot: true
        runAsUser: 10001
      containers:
        - name: gateway
          image: registry.example.com/ai/gateway:1.0.0
          command: ["uvicorn", "gateway.main:app", "--host", "0.0.0.0", "--port", "8510"]
          env:
            - name: GATEWAY_ROUTES
              value: /config/routes.json
            - name: GATEWAY_KEYFILE
              value: /secrets/api_keys.json
            - name: GATEWAY_LOG
              value: /logs/gateway.log
          ports:
            - name: http
              containerPort: 8510
          resources:
            requests:
              cpu: 100m
              memory: 128Mi
            limits:
              memory: 256Mi
          readinessProbe:
            httpGet:
              path: /gateway/health
              port: http
            periodSeconds: 5
          livenessProbe:
            httpGet:
              path: /gateway/health
              port: http
            periodSeconds: 10
          volumeMounts:
            - name: config
              mountPath: /config
              readOnly: true
            - name: secrets
              mountPath: /secrets
              readOnly: true
            - name: logs
              mountPath: /logs
      volumes:
        - name: config
          configMap:
            name: gateway-routes
        - name: secrets
          secret:
            secretName: gateway-keys
        - name: logs
          emptyDir: {}
---
apiVersion: v1
kind: Service
metadata:
  name: gateway
spec:
  selector:
    app: gateway
  ports:
    - name: http
      port: 8510
      targetPort: http

Writing k8s/gateway.yaml


Secret `gateway-keys`는 매니페스트 파일에 없습니다.  클러스터에 배포할 때 `kubectl create secret generic gateway-keys --from-file=api_keys.json=secrets/api_keys.json`으로 생성합니다. 키 파일은 폴더(`/secrets`)로 붙이고 로그 폴더는 `emptyDir`로 붙였습니다(이론 노트 2.4절). Step 5의 두 검사로 읽습니다.

In [ ]:
docs = [d for d in yaml.safe_load_all(Path("k8s/gateway.yaml").read_text(encoding="utf-8")) if d]
print("규격 오류 :", sum(len(schema_errors(d)) for d in docs), "건")
print("우리 규칙 :", check_manifests(docs, load_seconds=2) or "지적 없음")

규격 오류 : 0 건
우리 규칙 : 지적 없음


## 부록 실습 H. 롤링 업데이트 계획표

게이트웨이 노트 5.2절의 롤링 배포는 서버 한 대를 라우팅에서 빼고 새 버전으로 재시작하기를 되풀이했습니다. 디플로이먼트는 같은 일을 설정 둘로 합니다. `maxSurge`는 바꾸는 동안 원래 복제본 수보다 **더** 기동할 수 있는 파드 수이고 `maxUnavailable`은 원래 복제본 수보다 **덜** 준비돼 있어도 되는 파드 수입니다. 둘 다 기본값은 25%이고 파드 수로 바꿀 때 `maxSurge`는 올림, `maxUnavailable`은 내림합니다. GPU를 한 장씩 쓰는 서버에서 이 기본값이 어떻게 동작하는지 계산합니다.

In [51]:
import math


def rollout_plan(replicas, max_surge="25%", max_unavailable="25%", spare_gpus=0):
    """GPU 를 한 장씩 쓰는 파드의 롤링을 스텝마다 (옛 파드, 새 파드, 배치 대기, 빈 GPU) 로 계산합니다. (기록, 끝났는가)"""
    pct = lambda v, f: f(replicas * int(v[:-1]) / 100) if str(v).endswith("%") else int(v)
    surge, unavailable = pct(max_surge, math.ceil), pct(max_unavailable, math.floor)   # maxSurge 는 올림, maxUnavailable 은 내림
    old, new, pending, free, log = replicas, 0, 0, spare_gpus, []
    while surge + unavailable and (len(log) < 2 or log[-1] != log[-2]) and new < replicas:
        pending += max(0, min(replicas - new - pending, replicas + surge - old - new - pending))   # 새 파드를 만듭니다
        placed = min(pending, free)                                               # 빈 GPU 가 있는 만큼 배치됩니다
        pending, free, new = pending - placed, free - placed, new + placed
        removed = max(0, min(old, old + new - (replicas - unavailable)))           # 준비된 파드를 지키며 옛 파드를 줄입니다
        old, free = old - removed, free + removed
        log.append((old, new, pending, free))
    return log, new == replicas


for replicas in [2, 3, 4, 8]:
    log, done = rollout_plan(replicas)
    print(f"서버 {replicas}대, 빈 GPU 0장, 기본값 25% :", "끝남" if done else f"멈춤 (옛 {log[-1][0]}, 새 {log[-1][1]}, 배치 대기 {log[-1][2]})")

서버 2대, 빈 GPU 0장, 기본값 25% : 멈춤 (옛 2, 새 0, 배치 대기 1)
서버 3대, 빈 GPU 0장, 기본값 25% : 멈춤 (옛 3, 새 0, 배치 대기 1)
서버 4대, 빈 GPU 0장, 기본값 25% : 끝남
서버 8대, 빈 GPU 0장, 기본값 25% : 끝남


기본값으로는 서버가 세 대 이하일 때 롤링이 멈춥니다. `maxUnavailable`이 0으로 내림되어 옛 파드를 하나도 종료할 수 없는데 새 파드는 빈 GPU가 없어 배치되지 못하기 때문입니다.

![빈 GPU가 없으면 새 파드를 먼저 기동할 수 없으므로 옛 파드를 하나 먼저 종료하는 설정으로 바꿔야 롤링이 끝납니다.](lms-img:0)

*빈 GPU가 없으면 새 파드를 먼저 기동할 수 없으므로 옛 파드를 하나 먼저 종료하는 설정으로 바꿔야 롤링이 끝납니다.*

**실습** : GPU 서버 세 대가 빈 GPU 없이 실행 중입니다. 롤링이 끝나는 `maxSurge`와 `maxUnavailable`을 고르고 바꾸는 동안 요청을 받는 서버가 몇 대까지 줄어드는지 계획표에서 읽습니다.

In [52]:
# 실습 : 빈 GPU 없이 롤링이 끝나는 maxSurge 와 maxUnavailable 고르기
### START CODE HERE ### (maxSurge 와 maxUnavailable 값 2줄)
MY_SURGE = 0          # 1) 바꾸는 동안 원래 복제본 수보다 더 기동할 수 있는 파드 수 (예 : 0, 1, "25%")
MY_UNAVAILABLE = 1    # 2) 바꾸는 동안 원래 복제본 수보다 덜 준비돼 있어도 되는 파드 수
### END CODE HERE ###
log, done = rollout_plan(3, MY_SURGE, MY_UNAVAILABLE)
for i, (old, new, pending, free) in enumerate(log, 1):
    print(f"{i}스텝  옛 {old}  새 {new}  배치 대기 {pending}  빈 GPU {free}  -> 요청을 받는 서버 {old + new}대")
print("끝남" if done else "멈춤")

1스텝  옛 2  새 0  배치 대기 0  빈 GPU 1  -> 요청을 받는 서버 2대
2스텝  옛 1  새 1  배치 대기 0  빈 GPU 1  -> 요청을 받는 서버 2대
3스텝  옛 0  새 2  배치 대기 0  빈 GPU 1  -> 요청을 받는 서버 2대
4스텝  옛 0  새 3  배치 대기 0  빈 GPU 0  -> 요청을 받는 서버 3대
끝남


<details class="lms-answer">
<summary>풀이 보기</summary>

**빈칸 정답 코드 (`### START CODE HERE ###` 영역)**

```python
MY_SURGE = 0
MY_UNAVAILABLE = 1
```

`maxSurge: 0`, `maxUnavailable: 1`이면 옛 파드 하나를 먼저 종료해 GPU를 비우고 그 GPU에 새 파드를 기동하기를 되풀이합니다. 바꾸는 동안 요청을 받는 서버가 두 대로 줄어듭니다.

</details>

## 부록 실습 I.  kind 클러스터의 자동 복구와 GPU 부족

Step 7의 클러스터를 지우지 않은 상태에서 합니다. 파드의 상태를 쿠버네티스 파이썬 클라이언트로 읽는 함수부터 둡니다.

In [53]:
def pod_rows(pod_list):
    """V1PodList 를 (이름, 버전, 준비, 재시작 수, 상태) 행으로 바꿉니다."""
    rows = []
    for p in pod_list.items:
        statuses = p.status.container_statuses or []
        cs = statuses[0] if statuses else None
        rows.append((p.metadata.name, p.spec.containers[0].image.rsplit(":", 1)[-1],
                     bool(cs and cs.ready), cs.restart_count if cs else 0, p.status.phase))
    return rows


def ensure_cluster():
    """Step 7 의 클러스터와 디플로이먼트 qa 가 있는지 봅니다. 클러스터를 다시 만드는 일은 오래 걸려 안내만 출력합니다"""
    code, _ = sh("kubectl", "get", "deployment", "qa", check=False)
    if code != 0:
        print("실습 클러스터나 디플로이먼트 qa가 없습니다. Step 7의 로컬 터미널에서 적용을 마친 뒤 확인하세요.")
    return code == 0


if kind_ready() and ensure_cluster():
    from kubernetes import client, config
    config.load_kube_config(context="kind-d67")
    core, apps = client.CoreV1Api(), client.AppsV1Api()
    for row in pod_rows(core.list_namespaced_pod("default", label_selector="app=qa")):
        print("  ", row)

**자동 복구** : 파드 하나를 삭제하면 디플로이먼트가 모자란 파드 하나를 새로 생성합니다.

In [54]:
if not kind_ready():
    print("🖥 클러스터가 있는 기계에서 (Mac 터미널과 Windows PowerShell 에서 같습니다) :")
    print("   kubectl --context kind-d67 delete pod <파드 이름 하나>\n   kubectl --context kind-d67 get pods -l app=qa")
elif ensure_cluster():
    victim = pod_rows(core.list_namespaced_pod("default", label_selector="app=qa"))[0][0]
    sh("kubectl", "delete", "pod", victim)
    time.sleep(15)
    for row in pod_rows(core.list_namespaced_pod("default", label_selector="app=qa")):
        print("  ", row)

🖥 클러스터가 있는 기계에서 (Mac 터미널과 Windows PowerShell 에서 같습니다) :
   kubectl --context kind-d67 delete pod <파드 이름 하나>
   kubectl --context kind-d67 get pods -l app=qa


삭제한 파드 대신 이름이 다른 파드 하나가 새로 생성되어 있습니다.

**확장 리소스로 재현한 GPU 부족** : 쿠버네티스에는 노드에 아무 이름의 리소스를 몇 개 있다고 알려 주는 **확장 리소스**가 있습니다. 노드에 `example.com/gpu` 두 개를 알리고 서버 두 대가 한 개씩 쓰게 하면 빈 GPU가 없는 GPU 서버 두 대와 같아집니다.

In [55]:
if not kind_ready():
    print("🖥 Step 7 의 클러스터가 있는 기계에서 실행하세요")
elif ensure_cluster():
    node = core.list_node().items[0].metadata.name
    core.patch_node_status(node, [{"op": "add", "path": "/status/capacity/example.com~1gpu", "value": "2"}])
    print(node, "의 확장 리소스 :", core.read_node(node).status.allocatable.get("example.com/gpu"))

    patch = {"spec": {"progressDeadlineSeconds": 60,
                      "strategy": {"rollingUpdate": {"maxSurge": "25%", "maxUnavailable": "25%"}},
                      "template": {"spec": {"containers": [{"name": "qa", "resources": {
                          "limits": {"example.com/gpu": "1", "memory": "256Mi"}}}]}}}}
    apps.patch_namespaced_deployment("qa", "default", patch,
                                     _content_type="application/strategic-merge-patch+json")
    sh("kubectl", "rollout", "status", "deployment/qa", "--timeout=120s")
    sh("kubectl", "set", "image", "deployment/qa", "qa=qa-server:1.6.0")
    time.sleep(75)                                    # progressDeadlineSeconds 60 초를 넘겨 기다립니다
    for row in pod_rows(core.list_namespaced_pod("default", label_selector="app=qa")):
        print("  ", row)
    cond = [c for c in apps.read_namespaced_deployment("qa", "default").status.conditions if c.type == "Progressing"]
    print("Progressing :", cond[0].reason if cond else "(없음)")

🖥 Step 7 의 클러스터가 있는 기계에서 실행하세요


**과제** : 새 파드 하나가 `Pending`으로 남고 `Progressing`의 사유가 `ProgressDeadlineExceeded`인지 확인합니다. 그다음 `maxSurge: 0`, `maxUnavailable: 1`로 바꿔 롤링이 끝나는지 확인합니다. 결과는 부록 실습 H의 계획표와 비교합니다. 다 했으면 Step 9의 클러스터를 지우는 셀을 실행합니다.